In [ ]:
# ============================
# DMOFF — Full Jupyter Script (Reviewer-upgraded + Additional Metrics & Efficiency Logging)
# ============================
# Paste this whole cell in Jupyter and run.
# Make sure to change DATA_PATH to your local path.

# ---------- (Optional) Install required packages ----------
# Uncomment and run if packages are missing.
# !pip install numpy pandas matplotlib seaborn scikit-learn lightgbm optuna tensorflow odfpy pyexcel-ods3 shap joblib

# ---------- Imports ----------
import os
import warnings
warnings.filterwarnings("ignore")
import time
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import joblib
import shap
from datetime import timedelta
import psutil  # for memory if available
import pickle
import pathlib

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ---------- User-config ----------
DATA_PATH = r"E:\Abroad period research\Time-Series KD + Explainability\C3A9E300.ods"
TARGET_COL = "O3"
DATETIME_COL = "FECHA_HORA"
RESULTS_DIR = "results_dmoff_upgraded_with_metrics"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------- Helper functions ----------
def read_table(path):
    """Read .ods or .xlsx / .csv; try sensible engines."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".ods":
        try:
            df = pd.read_excel(path, engine="odf")
            return df
        except Exception as e:
            raise RuntimeError("Error reading .ods - install odfpy or convert to .xlsx. Exception: " + str(e))
    elif ext in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif ext == ".csv":
        return pd.read_csv(path)
    else:
        raise RuntimeError("Unsupported file extension: " + ext)

def create_time_features(df, datetime_col):
    df['hour'] = df[datetime_col].dt.hour
    df['day'] = df[datetime_col].dt.day
    df['weekday'] = df[datetime_col].dt.weekday
    df['month'] = df[datetime_col].dt.month
    # cyclic encodings
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['month_sin'] = np.sin(2*np.pi*(df['month']-1)/12)
    df['month_cos'] = np.cos(2*np.pi*(df['month']-1)/12)
    return df

def make_lags(df, cols, lags):
    for col in cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df

def train_test_chrono_split(df, test_size_hours):
    """Chronological split: last `test_size_hours` rows as test."""
    train = df.iloc[:-test_size_hours].copy()
    test = df.iloc[-test_size_hours:].copy()
    return train, test

# ---------- Additional metric helper functions (NEW, non-invasive) ----------
def smape(y_true, y_pred):
    """Symmetric mean absolute percentage error"""
    denom = (np.abs(y_true) + np.abs(y_pred))
    # avoid divide by zero
    mask = denom == 0
    denom[mask] = 1e-8
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def nse(y_true, y_pred):
    """Nash–Sutcliffe efficiency"""
    num = np.sum((y_true - y_pred)**2)
    denom = np.sum((y_true - np.mean(y_true))**2)
    if denom == 0:
        return np.nan
    return 1 - num/denom

def willmott_d(y_true, y_pred):
    """Willmott's index of agreement"""
    denom = np.sum((np.abs(y_pred - np.mean(y_true)) + np.abs(y_true - np.mean(y_true)))**2)
    num = np.sum((y_true - y_pred)**2)
    if denom == 0:
        return np.nan
    return 1 - num / denom

def model_file_size_mb(path):
    """Return file size in MB (if exists)"""
    try:
        return os.path.getsize(path) / (1024*1024)
    except Exception:
        return np.nan

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan

# ---------- Logging containers (NEW) ----------
metrics_log = {}   # will store runtime & size & performance per model/regime
train_timers = {}
model_sizes = {}
performance_summary = {}  # global metrics
per_regime_metrics = []

# ---------- 1) Load data ----------
print("Loading data from:", DATA_PATH)
df = read_table(DATA_PATH)
print("Raw columns:", df.columns.tolist())
df.columns = [c.strip() for c in df.columns]

# If DATETIME_COL missing, try to auto-detect
if DATETIME_COL not in df.columns:
    for c in df.columns:
        if "FECHA" in c.upper() or "HORA" in c.upper():
            DATETIME_COL = c
            print("Using datetime column:", DATETIME_COL)
            break

# Parse datetime
df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], dayfirst=True, errors='coerce')
if df[DATETIME_COL].isna().any():
    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL].astype(str), dayfirst=True, errors='coerce')
assert df[DATETIME_COL].notna().all(), "Some datetime values could not be parsed. Check your FECHA_HORA column."

df = df.sort_values(DATETIME_COL).reset_index(drop=True)

# rename simple columns if user uses longer names (common in their sample)
rename_map = {}
for col in df.columns:
    c = col.strip()
    if c.lower().startswith("dd"):
        rename_map[col] = "DD"
    if "NO2" in c.upper():
        rename_map[col] = "NO2"
    if "O3" == c.upper() or c.upper().endswith("-O3") or c.upper().startswith("O3"):
        rename_map[col] = "O3"
    if "PM10" in c.upper():
        rename_map[col] = "PM10"
    if "TMP" in c.upper() or "TEMP" in c.upper():
        rename_map[col] = "TMP"
    if "VV" in c.upper():
        rename_map[col] = "VV"
if rename_map:
    df = df.rename(columns=rename_map)
print("After renaming, columns:", df.columns.tolist())

assert TARGET_COL in df.columns, f"{TARGET_COL} not found in data columns."

# ---------- 2) Feature engineering ----------
df = create_time_features(df, DATETIME_COL)

# numeric columns used in clustering and modeling (only keep existing)
# model_cols = ["CO", "DD", "NO2", "PM10", "TMP", "VV"]
model_cols = ["DD", "NO2", "PM10", "TMP", "VV"]
model_cols = [c for c in model_cols if c in df.columns]

# create lags for O3 and main predictors
lags = [1,2,3,24]   # 1,2,3 hour and 24-hour lag
lags_for = [TARGET_COL] + model_cols
df = make_lags(df, lags_for, lags)
df = df.dropna().reset_index(drop=True)
print("Data shape after lags and dropna:", df.shape)

# ---------- 3) Discover regimes via clustering (GMM) - TRAIN-ONLY, with smoothing & validation ----------
cluster_features = [c for c in model_cols + ['hour_sin','hour_cos','month_sin','month_cos'] if c in df.columns]
print("Cluster features:", cluster_features)

# We'll split chronologically first to avoid leakage for the GMM selection
test_fraction = 0.2
n_test = int(len(df) * test_fraction)
train_df_all, test_df_all = train_test_chrono_split(df, test_size_hours=n_test)
print("Clustering will be fit on train only. Train rows:", len(train_df_all), "Test rows:", len(test_df_all))

# Fit scaler on train clustering features only
scaler_cluster = StandardScaler().fit(train_df_all[cluster_features].values)
Xc_train = scaler_cluster.transform(train_df_all[cluster_features].values)
Xc_test = scaler_cluster.transform(test_df_all[cluster_features].values)

# Choose number of clusters using BIC on train only
def select_gmm_k_on_train(X_train, k_min=2, k_max=6, random_state=RANDOM_SEED):
    best_bic = np.inf
    best_k = k_min
    best_model = None
    for k in range(k_min, k_max+1):
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=random_state, n_init=5)
        gmm.fit(X_train)
        bic = gmm.bic(X_train)
        print(f"GMM k={k} BIC={bic:.1f}")
        if bic < best_bic:
            best_bic = bic
            best_k = k
            best_model = gmm
    return best_k, best_model

print("Selecting number of regimes (GMM) on train only...")
best_k, gmm = select_gmm_k_on_train(Xc_train, k_min=2, k_max=5)
print("Selected regimes (k):", best_k)

# Assign regimes: predict on train & test using scaler and trained GMM
train_df_all['regime'] = gmm.predict(Xc_train)
test_df_all['regime'] = gmm.predict(Xc_test)

# Optional temporal smoothing to enforce regime persistence (small window)
SMOOTH_REGIMES = True
SMOOTH_WINDOW = 3  # odd integer
if SMOOTH_REGIMES:
    train_df_all['regime'] = train_df_all['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)
    test_df_all['regime'] = test_df_all['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)

# Compute clustering validity metrics on train
try:
    sil = silhouette_score(Xc_train, train_df_all['regime'])
    dbi = davies_bouldin_score(Xc_train, train_df_all['regime'])
    print(f"Clustering validity (train): Silhouette = {sil:.4f}, Davies-Bouldin = {dbi:.4f}")
    metrics_log['clustering'] = {'silhouette': float(sil), 'davies_bouldin': float(dbi)}
except Exception as e:
    print("Could not compute silhouette/DBI:", e)

# Create full df regimes by applying scaler and gmm predict to entire df (no refit)
Xc_full = scaler_cluster.transform(df[cluster_features].values)
df['regime'] = gmm.predict(Xc_full)
if SMOOTH_REGIMES:
    df['regime'] = df['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)

# Regime summary table (means)
regime_summary = df.groupby('regime')[cluster_features].mean().round(3)
print("Regime summary (means):")
display(regime_summary)

# plot regime counts
plt.figure(figsize=(6,3))
df['regime'].value_counts().sort_index().plot(kind='bar', title='Regime counts')
plt.show()

# Persist cluster scaler & gmm
joblib.dump(scaler_cluster, os.path.join(RESULTS_DIR, "scaler_cluster.pkl"))
joblib.dump(gmm, os.path.join(RESULTS_DIR, "gmm_regimes.pkl"))

# ---------- 4) Chronological train/test split for full modelling ----------
# We'll now use train_df_all (already computed) and test_df_all for training models
train_df = train_df_all.copy().reset_index(drop=True)
test_df = test_df_all.copy().reset_index(drop=True)
print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
metrics_log['data_split'] = {'n_total': int(len(df)), 'n_train': int(len(train_df)), 'n_test': int(len(test_df)),
                            'train_fraction': float(len(train_df)/len(df)), 'test_fraction': float(len(test_df)/len(df))}

# ---------- 5) Per-regime model training ----------
# Common feature set for tabular models
tab_features = []
# features include lags for each variable in lags_for (like O3_lag1...) plus meteorological/time features
for c in df.columns:
    # if column starts with any lag prefix or matches meta features
    if any([c.startswith(f"{col}_lag") for col in lags_for]) or c in model_cols + ['hour','weekday','month','hour_sin','hour_cos','month_sin','month_cos']:
        tab_features.append(c)
# ensure target not in features
tab_features = [c for c in tab_features if c != TARGET_COL and c != DATETIME_COL and c != 'regime']
print("Tabular features used (sample):", tab_features[:30])

# Setup storage
regime_models = {}
regimes_unique = sorted(train_df['regime'].unique())
print("Regimes to train:", regimes_unique)

# ---------- 5.A) Train LightGBM per regime with Optuna ----------
def fit_lgb_for_regime(Xtrain, ytrain, Xval, yval, n_trials=25, seed=RANDOM_SEED):
    def objective(trial):
        param = {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "seed": seed,
            "num_leaves": trial.suggest_int("num_leaves", 16, 128),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        }

        dtrain = lgb.Dataset(Xtrain, label=ytrain)
        dvalid = lgb.Dataset(Xval, label=yval)

        bst = lgb.train(
            param,
            dtrain,
            valid_sets=[dvalid],
            num_boost_round=1000,
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(0)
            ]
        )

        preds = bst.predict(Xval, num_iteration=bst.best_iteration)
        return mean_squared_error(yval, preds, squared=False)  # RMSE

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objectine, n_trials=n_trials, show_progress_bar=False)

    # Fit final model on combined train+val using best params
    best_params = study.best_params
    best_params.update({
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "seed": seed
    })

    dtrain_all = lgb.Dataset(np.vstack([Xtrain, Xval]), label=np.concatenate([ytrain, yval]))
    final = lgb.train(
        best_params,
        dtrain_all,
        num_boost_round=200  # keep reasonable
    )

    # store iteration info
    study.best_trial.user_attrs["best_iteration"] = final.current_iteration() if hasattr(final, "current_iteration") else 200
    return final, study

# Timing variables for LGB training
lgb_total_train_time = 0.0
lgb_regime_times = {}

for r in regimes_unique:
    print("Training LGB for regime:", r)
    start_r = time.time()
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < 120:
        print(f"Regime {r} has too few samples ({len(df_r)}). Skipping per-regime LGB (will use global fallback later).")
        continue
    split_idx = int(0.8 * len(df_r))
    train_r = df_r.iloc[:split_idx]
    val_r = df_r.iloc[split_idx:]
    Xtrain = train_r[tab_features].values
    ytrain = train_r[TARGET_COL].values
    Xval = val_r[tab_features].values
    yval = val_r[TARGET_COL].values
    scaler = StandardScaler().fit(np.vstack([Xtrain, Xval]))
    Xtrain_s = scaler.transform(Xtrain)
    Xval_s = scaler.transform(Xval)
    print("  Tuning LightGBM (Optuna)...")
    try:
        t0 = time.time()
        lgb_model, study = fit_lgb_for_regime(Xtrain_s, ytrain, Xval_s, yval, n_trials=25)
        t1 = time.time()
        elapsed = t1 - t0
        lgb_total_train_time += elapsed
        lgb_regime_times[r] = elapsed
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['lgb_model'] = lgb_model
        regime_models[r]['lgb_scaler'] = scaler
        joblib.dump(study, os.path.join(RESULTS_DIR, f"optuna_lgb_regime_{r}.pkl"))
        # Save model and record size
        lgb_path = os.path.join(RESULTS_DIR, f"lgb_regime_{r}.txt")
        lgb_model.save_model(lgb_path)
        size_mb = model_file_size_mb(lgb_path)
        model_sizes[f"lgb_regime_{r}"] = size_mb
        train_timers[f"lgb_regime_{r}"] = elapsed
        print("  LightGBM done. best RMSE (val):", study.best_value, "train time(s):", round(elapsed,2), "size(MB):", round(size_mb,3))
    except Exception as e:
        print("  LightGBM training failed for regime", r, "due to:", e)

# Train a global fallback LightGBM on all train data for regimes with too few samples
print("Training global fallback LightGBM on whole train set (for small regimes)...")
lgb_global = None
scaler_glob = None
lgb_global_train_time = None
try:
    t0_glob = time.time()
    split_idx = int(0.9 * len(train_df))
    train_glob = train_df.iloc[:split_idx]
    val_glob = train_df.iloc[split_idx:]
    Xtrain_g = train_glob[tab_features].values
    ytrain_g = train_glob[TARGET_COL].values
    Xval_g = val_glob[tab_features].values
    yval_g = val_glob[TARGET_COL].values
    scaler_glob = StandardScaler().fit(np.vstack([Xtrain_g, Xval_g]))
    Xtrain_gs = scaler_glob.transform(Xtrain_g)
    Xval_gs = scaler_glob.transform(Xval_g)
    lgb_global, study_glob = fit_lgb_for_regime(Xtrain_gs, ytrain_g, Xval_gs, yval_g, n_trials=20)
    t1_glob = time.time()
    lgb_global_train_time = t1_glob - t0_glob
    joblib.dump(lgb_global, os.path.join(RESULTS_DIR, "lgb_global.pkl"))
    joblib.dump(scaler_glob, os.path.join(RESULTS_DIR, "scaler_lgb_global.pkl"))
    # save text model
    lgb_global_path = os.path.join(RESULTS_DIR, "lgb_global.txt")
    lgb_global.save_model(lgb_global_path)
    size_glob = model_file_size_mb(lgb_global_path)
    model_sizes["lgb_global"] = size_glob
    train_timers["lgb_global"] = lgb_global_train_time
    print("Global LGB trained. best RMSE:", study_glob.best_value, "time(s):", round(lgb_global_train_time,2), "size(MB):", round(size_glob,3))
except Exception as e:
    print("Global LightGBM training failed:", e)

# ---------- 5.B) Train small LSTM per regime ----------
SEQ_LEN = 24  # use last 24 hours to predict next hour

def make_sequences(df_reg, seq_len, feature_cols, target_col):
    Xs, ys = [], []
    arr = df_reg[feature_cols].values
    tar = df_reg[target_col].values
    for i in range(seq_len, len(df_reg)):
        Xs.append(arr[i-seq_len:i])
        ys.append(tar[i])
    return np.array(Xs), np.array(ys)

lstm_histories = {}
lstm_train_times = {}
lstm_total_time = 0.0

for r in regimes_unique:
    print("Training LSTM for regime:", r)
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < SEQ_LEN + 30:
        print(f"  Too few samples ({len(df_r)}). Skipping LSTM for regime {r}.")
        continue
    seq_features = [c for c in tab_features]  # use same tab features in sequence
    scaler_seq = MinMaxScaler()
    scaler_seq.fit(df_r[seq_features].values)
    arr_scaled = scaler_seq.transform(df_r[seq_features].values)
    df_r_scaled = df_r.copy()
    df_r_scaled[seq_features] = arr_scaled
    Xs, ys = make_sequences(df_r_scaled, SEQ_LEN, seq_features, TARGET_COL)
    split = int(0.8 * len(Xs))
    Xtrain, Xval = Xs[:split], Xs[split:]
    ytrain, yval = ys[:split], ys[split:]
    def build_lstm_model(n_features, lr=1e-3, units=64, dropout=0.2):
        inp = keras.Input(shape=(SEQ_LEN, n_features))
        x = layers.LSTM(units, return_sequences=False)(inp)
        x = layers.Dropout(dropout)(x)
        out = layers.Dense(1)(x)
        model = keras.Model(inputs=inp, outputs=out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss='mse')
        return model
    def optuna_lstm_objective(trial):
        units = trial.suggest_int("units", 16, 128)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
        model = build_lstm_model(Xtrain.shape[2], lr=lr, units=units, dropout=dropout)
        early = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')
        h = model.fit(Xtrain, ytrain, validation_data=(Xval, yval), epochs=60, batch_size=32, callbacks=[early], verbose=0)
        val_rmse = np.sqrt(min(h.history['val_loss']))
        return val_rmse
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    n_trials = 20
    try:
        t0 = time.time()
        study.optimize(optuna_lstm_objective, n_trials=n_trials, show_progress_bar=False)
        t1 = time.time()
        elapsed = t1 - t0
        lstm_total_time += elapsed
        lstm_train_times[r] = elapsed
        print("  Best LSTM RMSE (val):", study.best_value, "t(s):", round(elapsed,2))
        best = study.best_params
        final_model = build_lstm_model(Xs.shape[2], lr=best.get('lr',1e-3), units=best.get('units',64), dropout=best.get('dropout',0.2))
        early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='loss')
        t0f = time.time()
        final_model.fit(Xs, ys, epochs=100, batch_size=32, callbacks=[early], verbose=0)
        t1f = time.time()
        fit_elapsed = t1f - t0f
        total_elapsed = elapsed + fit_elapsed
        lstm_train_times[r] = total_elapsed
        lstm_histories[r] = study.best_value
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['lstm_model'] = final_model
        regime_models[r]['lstm_scaler'] = scaler_seq
        # save model (h5) and size
        lstm_path = os.path.join(RESULTS_DIR, f"lstm_regime_{r}.h5")
        try:
            final_model.save(lstm_path)
        except Exception:
            # fallback to pickle for keras (may not be ideal)
            with open(os.path.join(RESULTS_DIR, f"lstm_regime_{r}.pkl"), "wb") as f:
                pickle.dump(final_model, f)
            lstm_path = os.path.join(RESULTS_DIR, f"lstm_regime_{r}.pkl")
        size_mb = model_file_size_mb(lstm_path)
        model_sizes[f"lstm_regime_{r}"] = size_mb
        train_timers[f"lstm_regime_{r}"] = total_elapsed
        joblib.dump(study, os.path.join(RESULTS_DIR, f"optuna_lstm_regime_{r}.pkl"))
        print(f"  Final LSTM fit time(s): {round(fit_elapsed,2)}, total(s): {round(total_elapsed,2)}, size(MB): {round(size_mb,3)}")
    except Exception as e:
        print("  LSTM training failed for regime", r, "due to:", e)

# ---------- 6) Train regime classifier (meta-controller) ----------
# The classifier predicts the regime at time t given current features (trained on train set only)
clf_features = cluster_features + [TARGET_COL] + [f"{TARGET_COL}_lag{l}" for l in lags]
clf_features = [c for c in clf_features if c in train_df.columns]
print("Regime classifier features:", clf_features)

Xclf = train_df[clf_features].values
yclf = train_df['regime'].values
clf_scaler = StandardScaler().fit(Xclf)
Xclf_s = clf_scaler.transform(Xclf)
t0_clf = time.time()
rfc = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED)
rfc.fit(Xclf_s, yclf)
t1_clf = time.time()
clf_train_time = t1_clf - t0_clf
joblib.dump(rfc, os.path.join(RESULTS_DIR, "regime_classifier.pkl"))
joblib.dump(clf_scaler, os.path.join(RESULTS_DIR, "regime_clf_scaler.pkl"))
# record size/time
rfc_path = os.path.join(RESULTS_DIR, "regime_classifier.pkl")
model_sizes["regime_classifier_rf"] = model_file_size_mb(rfc_path)
train_timers["regime_classifier_rf"] = clf_train_time
print("Regime classifier trained. Train accuracy (on scaled train):", rfc.score(Xclf_s, yclf), "time(s):", round(clf_train_time,2), "size(MB):", round(model_sizes["regime_classifier_rf"],3))

# ---------- 7) Attention-style meta-blender ----------
from tensorflow.keras import backend as K

def build_blender(input_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    w = layers.Dense(2, activation='softmax')(x)  # outputs two weights summing to 1
    model = keras.Model(inputs=inp, outputs=w)
    # we'll use custom training loop below
    return model

# Prepare blender training data: use last 10% of train_df (chronological)
blend_df = train_df.iloc[int(0.9*len(train_df)):].copy().reset_index(drop=True)
blend_features = tab_features  # use tab features as input to blender
X_blend = []
Y_blend = []
preds_lgb_all = []
preds_lstm_all = []

# We'll use the global fallback LGB/scaler if per-regime LGB missing
lgb_global_exists = os.path.exists(os.path.join(RESULTS_DIR, "lgb_global.pkl"))
if lgb_global_exists:
    try:
        lgb_global = joblib.load(os.path.join(RESULTS_DIR, "lgb_global.pkl"))
        scaler_glob = joblib.load(os.path.join(RESULTS_DIR, "scaler_lgb_global.pkl"))
    except:
        lgb_global = None
        scaler_glob = None
else:
    lgb_global = None
    scaler_glob = None

for idx, row in blend_df.reset_index(drop=True).iterrows():
    r = int(row['regime'])
    # LGB prediction
    if 'lgb_model' in regime_models.get(r, {}):
        scaler_lgb = regime_models[r]['lgb_scaler']
        lgb_model = regime_models[r]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_global.predict(X_row)[0])
    else:
        pred_lgb = float(row[TARGET_COL])  # fallback to persistence

    # LSTM prediction: build sequence from training data only (NO DATA LEAKAGE)
    if 'lstm_model' in regime_models.get(r, {}):
        lstm_model = regime_models[r]['lstm_model']
        scaler_seq = regime_models[r]['lstm_scaler']
        dt = row[DATETIME_COL]
        # Use training data only for LSTM sequences - NO DATA LEAKAGE
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==r)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values)
            seq = seq.reshape(1, SEQ_LEN, len(tab_features))
            try:
                pred_lstm = float(lstm_model.predict(seq)[0,0])
            except:
                pred_lstm = pred_lgb
        else:
            pred_lstm = pred_lgb
    else:
        pred_lstm = pred_lgb

    preds_lgb_all.append(pred_lgb)
    preds_lstm_all.append(pred_lstm)
    X_blend.append(row[blend_features].values)
    Y_blend.append(row[TARGET_COL])

X_blend = np.array(X_blend)
Y_blend = np.array(Y_blend)
preds_lgb_all = np.array(preds_lgb_all)
preds_lstm_all = np.array(preds_lstm_all)

scaler_blend = StandardScaler().fit(X_blend)
X_blend_scaled = scaler_blend.transform(X_blend)
X_blend_in = np.hstack([X_blend_scaled, preds_lgb_all.reshape(-1,1), preds_lstm_all.reshape(-1,1)])
blender_input_dim = X_blend_in.shape[1]
print("Blender input dim:", blender_input_dim)

def blender_train_end_to_end(Xin, predsA, predsB, Y, batch=32, epochs=60):
    # Build model that outputs weights; we'll train with a custom loop minimizing MSE between weighted preds and Y
    inp = keras.Input(shape=(Xin.shape[1],))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    w = layers.Dense(2, activation='softmax')(x)
    model = keras.Model(inputs=inp, outputs=w)
    optimizer = keras.optimizers.Adam()
    # prepare dataset
    N = len(Xin)
    for epoch in range(epochs):
        idxs = np.arange(N)
        np.random.shuffle(idxs)
        epoch_losses = []
        for i in range(0, N, batch):
            batch_idx = idxs[i:i+batch]
            Xin_b = Xin[batch_idx]
            predsA_b = predsA[batch_idx]
            predsB_b = predsB[batch_idx]
            Y_b = Y[batch_idx]
            with tf.GradientTape() as tape:
                W_b = model(Xin_b, training=True)  # shape (b,2)
                combined = W_b[:,0]*predsA_b + W_b[:,1]*predsB_b
                loss = tf.reduce_mean(tf.square(combined - Y_b))
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            epoch_losses.append(float(loss))
        if epoch % 10 == 0 or epoch == epochs-1:
            print(f"Blender epoch {epoch}, loss {np.mean(epoch_losses):.6f}")
    return model

print("Training blender network (end-to-end)...")
t0_blend = time.time()
blender_model = blender_train_end_to_end(X_blend_in, preds_lgb_all, preds_lstm_all, Y_blend, batch=32, epochs=60)
t1_blend = time.time()
blender_train_time = t1_blend - t0_blend
# save blender & scaler
joblib.dump(scaler_blend, os.path.join(RESULTS_DIR, "scaler_blend.pkl"))
blender_path = os.path.join(RESULTS_DIR, "blender_model.h5")
try:
    blender_model.save(blender_path)
except Exception:
    # fallback to pickle
    with open(os.path.join(RESULTS_DIR, "blender_model.pkl"), "wb") as f:
        pickle.dump(blender_model, f)
    blender_path = os.path.join(RESULTS_DIR, "blender_model.pkl")
model_sizes["blender_model"] = model_file_size_mb(blender_path)
train_timers["blender_model"] = blender_train_time
print("Blender trained. time(s):", round(blender_train_time,2), "size(MB):", round(model_sizes["blender_model"],3))

# ---------- 8) Prediction pipeline on test set ----------
def predict_one_row(row):
    """Given a row from test_df, predict O3 for next hour using DMOFF pipeline."""
    # predict regime for current row using regime classifier (trained on train only)
    xf = row[clf_features].values.reshape(1,-1)
    xf_s = clf_scaler.transform(xf)
    pred_reg = int(rfc.predict(xf_s)[0])

    # LGB prediction (per-regime or global fallback)
    if 'lgb_model' in regime_models.get(pred_reg, {}):
        scaler_lgb = regime_models[pred_reg]['lgb_scaler']
        lgb_model = regime_models[pred_reg]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_global.predict(X_row)[0])
    else:
        pred_lgb = float(row[TARGET_COL])

    # LSTM prediction - USE TRAINING DATA ONLY (NO DATA LEAKAGE)
    if 'lstm_model' in regime_models.get(pred_reg, {}):
        lstm_model = regime_models[pred_reg]['lstm_model']
        scaler_seq = regime_models[pred_reg]['lstm_scaler']
        dt = row[DATETIME_COL]
        # Use training data only for LSTM sequences - NO DATA LEAKAGE
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==pred_reg)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values).reshape(1, SEQ_LEN, len(tab_features))
            try:
                pred_lstm = float(lstm_model.predict(seq)[0,0])
            except:
                pred_lstm = pred_lgb
        else:
            pred_lstm = pred_lgb
    else:
        pred_lstm = pred_lgb

    # Build blender input
    Xb_raw = row[blend_features].values.reshape(1,-1)
    Xb_scaled = scaler_blend.transform(Xb_raw)
    Xb_in = np.hstack([Xb_scaled, np.array([[pred_lgb, pred_lstm]])])
    w = blender_model.predict(Xb_in)[0]  # two weights
    pred = w[0]*pred_lgb + w[1]*pred_lstm
    return pred, pred_reg, pred_lgb, pred_lstm, w

# Run predictions across test set with timing
y_true = []
y_pred = []
pred_details = []
pred_times = []
start_all_preds = time.time()
for idx, row in test_df.iterrows():
    t0p = time.time()
    p, preg, plgb, plstm, w = predict_one_row(row)
    t1p = time.time()
    pred_times.append(t1p - t0p)
    y_true.append(row[TARGET_COL])
    y_pred.append(p)
    pred_details.append((preg, plgb, plstm, w))
end_all_preds = time.time()
total_pred_time = end_all_preds - start_all_preds
avg_pred_time = np.mean(pred_times) if pred_times else np.nan
pred_percentiles = np.percentile(pred_times, [50, 90, 95]) if pred_times else [np.nan, np.nan, np.nan]
metrics_log['prediction_time'] = {'total_s': float(total_pred_time), 'avg_per_sample_s': float(avg_pred_time),
                                 'p50_s': float(pred_percentiles[0]), 'p90_s': float(pred_percentiles[1]), 'p95_s': float(pred_percentiles[2])}

y_true = np.array(y_true); y_pred = np.array(y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
mape = None
try:
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100.0
except Exception:
    # fallback MAPE
    mape = np.mean(np.abs((y_true - y_pred) / (np.where(y_true == 0, 1e-8, y_true)))) * 100.0
smape_val = smape(y_true, y_pred)
r2 = safe_r2(y_true, y_pred)
nse_val = nse(y_true, y_pred)
willmott = willmott_d(y_true, y_pred)
median_ae = median_absolute_error(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}, MAE: {mae:.3f}, MAPE%: {mape:.3f}, sMAPE%: {smape_val:.3f}")
print(f"R2: {r2:.3f}, NSE: {nse_val:.3f}, Willmott d: {willmott:.3f}, MedianAE: {median_ae:.3f}")

# Save predictions
test_df_out = test_df.copy()
test_df_out['pred_dmoff'] = y_pred
test_df_out['obs'] = y_true
test_df_out.to_csv(os.path.join(RESULTS_DIR, "test_predictions_dmoff.csv"), index=False)

# ---------- 9) Diagnostics, per-regime metrics and plots ----------
# Per-regime performance with added metrics
test_df_out['error'] = test_df_out['pred_dmoff'] - test_df_out['obs']
per_regime_rows = []
for r in sorted(test_df_out['regime'].unique()):
    df_r = test_df_out[test_df_out['regime']==r]
    if len(df_r) > 0:
        yt = df_r['obs'].values
        yp = df_r['pred_dmoff'].values
        rmse_r = np.sqrt(mean_squared_error(yt, yp))
        mae_r = mean_absolute_error(yt, yp)
        # mape, smape, r2, nse, willmott, median
        try:
            mape_r = mean_absolute_percentage_error(yt, yp) * 100.0
        except Exception:
            mape_r = np.mean(np.abs((yt - yp) / (np.where(yt == 0, 1e-8, yt)))) * 100.0
        smape_r = smape(yt, yp)
        r2_r = safe_r2(yt, yp)
        nse_r = nse(yt, yp)
        will_r = willmott_d(yt, yp)
        median_ae_r = median_absolute_error(yt, yp)
        per_regime_rows.append({
            'regime': int(r),
            'n': len(df_r),
            'rmse': float(rmse_r),
            'mae': float(mae_r),
            'mape_pct': float(mape_r),
            'smape_pct': float(smape_r),
            'r2': float(r2_r) if not np.isnan(r2_r) else None,
            'nse': float(nse_r) if not np.isnan(nse_r) else None,
            'willmott_d': float(will_r) if not np.isnan(will_r) else None,
            'median_ae': float(median_ae_r)
        })
        print(f"Regime {r}: N={len(df_r)} RMSE={rmse_r:.3f}, MAE={mae_r:.3f}, MAPE%={mape_r:.3f}, sMAPE%={smape_r:.3f}, R2={r2_r:.3f}")

per_regime_df = pd.DataFrame(per_regime_rows)
per_regime_df.to_csv(os.path.join(RESULTS_DIR, "per_regime_metrics.csv"), index=False)

# ---------- 9.B) INDIVIDUAL MODEL PERFORMANCE COMPARISON ----------
print("\n" + "="*60)
print("INDIVIDUAL MODEL PERFORMANCE COMPARISON")
print("="*60)

# Store individual model predictions
test_df_out['pred_lgb_only'] = np.nan
test_df_out['pred_lstm_only'] = np.nan
test_df_out['pred_regime'] = np.nan

# Get individual model predictions
for idx, row in test_df_out.iterrows():
    r = int(row['regime'])
    
    # LightGBM only prediction
    if 'lgb_model' in regime_models.get(r, {}):
        scaler_lgb = regime_models[r]['lgb_scaler']
        lgb_model = regime_models[r]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_lgb_only'] = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_lgb_only'] = float(lgb_global.predict(X_row)[0])
    
    # LSTM only prediction (with the data leakage fix!)
    if 'lstm_model' in regime_models.get(r, {}):
        lstm_model = regime_models[r]['lstm_model']
        scaler_seq = regime_models[r]['lstm_scaler']
        dt = row[DATETIME_COL]
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==r)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values).reshape(1, SEQ_LEN, len(tab_features))
            try:
                test_df_out.at[idx, 'pred_lstm_only'] = float(lstm_model.predict(seq)[0,0])
            except:
                test_df_out.at[idx, 'pred_lstm_only'] = test_df_out.at[idx, 'pred_lgb_only']
    
    # Store the predicted regime
    test_df_out.at[idx, 'pred_regime'] = r

# Calculate metrics for each model
models_comparison = []

# Global performance for each model
for model_name, pred_col in [('DMOFF (Hybrid)', 'pred_dmoff'), 
                            ('LightGBM Only', 'pred_lgb_only'), 
                            ('LSTM Only', 'pred_lstm_only')]:
    
    if pred_col not in test_df_out.columns or test_df_out[pred_col].isna().all():
        continue
        
    mask = test_df_out[pred_col].notna()
    y_true_model = test_df_out.loc[mask, 'obs'].values
    y_pred_model = test_df_out.loc[mask, pred_col].values
    
    if len(y_true_model) > 0:
        rmse_m = np.sqrt(mean_squared_error(y_true_model, y_pred_model))
        mae_m = mean_absolute_error(y_true_model, y_pred_model)
        r2_m = safe_r2(y_true_model, y_pred_model)
        smape_m = smape(y_true_model, y_pred_model)
        
        models_comparison.append({
            'model': model_name,
            'n_samples': len(y_true_model),
            'rmse': rmse_m,
            'mae': mae_m,
            'r2': r2_m,
            'smape_pct': smape_m
        })
        
        print(f"{model_name:20} | RMSE: {rmse_m:.3f} | MAE: {mae_m:.3f} | R²: {r2_m:.3f} | sMAPE: {smape_m:.3f}%")

# Per-regime performance for each model
print("\n--- PER-REGIME COMPARISON ---")
for r in sorted(test_df_out['regime'].unique()):
    df_r = test_df_out[test_df_out['regime']==r]
    if len(df_r) == 0:
        continue
        
    print(f"\nRegime {r} (N={len(df_r)}):")
    
    for model_name, pred_col in [('DMOFF', 'pred_dmoff'), 
                                ('LightGBM', 'pred_lgb_only'), 
                                ('LSTM', 'pred_lstm_only')]:
        
        if pred_col not in df_r.columns or df_r[pred_col].isna().all():
            continue
            
        mask = df_r[pred_col].notna()
        if mask.sum() == 0:
            continue
            
        yt = df_r.loc[mask, 'obs'].values
        yp = df_r.loc[mask, pred_col].values
        
        rmse_r = np.sqrt(mean_squared_error(yt, yp))
        mae_r = mean_absolute_error(yt, yp)
        
        print(f"  {model_name:10} | RMSE: {rmse_r:.3f} | MAE: {mae_r:.3f}")

# Save comparison results
comparison_df = pd.DataFrame(models_comparison)
comparison_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_metrics.csv"), index=False)
test_df_out.to_csv(os.path.join(RESULTS_DIR, "test_predictions_with_individual_models.csv"), index=False)

print(f"\nComparison results saved to: model_comparison_metrics.csv")

# Plot overall obs vs pred
plt.figure(figsize=(14,5))
plt.plot(test_df_out[DATETIME_COL], test_df_out['obs'], label='obs', alpha=0.8)
plt.plot(test_df_out[DATETIME_COL], test_df_out['pred_dmoff'], label='DMOFF pred', alpha=0.8)
plt.legend(); plt.title("Observed vs DMOFF predictions (test)"); plt.xticks(rotation=25); plt.show()

# Feature importance for a representative regime (LGB)
rep_reg = regimes_unique[0] if regimes_unique else 0
if 'lgb_model' in regime_models.get(rep_reg, {}):
    print("LightGBM feature importance (regime {})".format(rep_reg))
    imp = regime_models[rep_reg]['lgb_model'].feature_importance(importance_type='gain')
    feat = tab_features
    fi = pd.DataFrame({'feature': feat, 'gain': imp}).sort_values('gain', ascending=False).head(20)
    display(fi)
    fi.plot(kind='barh', x='feature', y='gain', legend=False, title=f"LGB feature importance regime {rep_reg}"); plt.show()

# Global SHAP for LGB (may be heavy); only if shap installed and small data
try:
    if 'lgb_model' in regime_models.get(rep_reg, {}):
        expl = shap.TreeExplainer(regime_models[rep_reg]['lgb_model'])
        sampleX = regime_models[rep_reg]['lgb_scaler'].transform(train_df[train_df['regime']==rep_reg][tab_features].values[:200])
        shap_vals = expl.shap_values(sampleX)
        shap.summary_plot(shap_vals, sampleX, feature_names=tab_features, show=True)
except Exception as e:
    print("SHAP skipped due to:", e)

# ---------- 10) Efficiency & performance summary (NEW) ----------
# Collect global performance metrics
performance_summary = {
    'rmse': float(rmse),
    'mae': float(mae),
    'mape_pct': float(mape),
    'smape_pct': float(smape_val),
    'r2': float(r2) if not np.isnan(r2) else None,
    'nse': float(nse_val) if not np.isnan(nse_val) else None,
    'willmott_d': float(willmott) if not np.isnan(willmott) else None,
    'median_ae': float(median_ae),
    'n_test_samples': int(len(test_df)),
    'total_prediction_time_s': float(total_pred_time),
    'avg_prediction_time_per_sample_s': float(avg_pred_time)
}
# Merge with metrics_log
metrics_log['performance'] = performance_summary
metrics_log['model_sizes_mb'] = model_sizes
metrics_log['train_times_s'] = train_timers
metrics_log['lgb_per_regime_train_times'] = {str(k): float(v) for k,v in lgb_regime_times.items()}
metrics_log['lstm_per_regime_train_times'] = {str(k): float(v) for k,v in lstm_train_times.items()}
metrics_log['blender_train_time_s'] = float(blender_train_time)

# Save metrics_log as JSON
with open(os.path.join(RESULTS_DIR, "metrics_log.json"), "w") as f:
    json.dump(metrics_log, f, indent=2)

# Write a consolidated CSV with global and summary info
global_metrics_df = pd.DataFrame([performance_summary])
global_metrics_df.to_csv(os.path.join(RESULTS_DIR, "global_performance_metrics.csv"), index=False)

# Save per-regime metrics (already saved above), and a combined report
report = {
    'global_metrics': performance_summary,
    'per_regime': per_regime_rows,
    'model_sizes_mb': model_sizes,
    'train_times_s': train_timers,
    'clustering_metrics': metrics_log.get('clustering', {})
}
with open(os.path.join(RESULTS_DIR, "dmoff_report_summary.json"), "w") as f:
    json.dump(report, f, indent=2)

# ---------- 11) Save everything ----------
joblib.dump(regime_models, os.path.join(RESULTS_DIR, "regime_models.pkl"))
joblib.dump(rfc, os.path.join(RESULTS_DIR, "regime_classifier_rf.pkl"))
joblib.dump(scaler_cluster, os.path.join(RESULTS_DIR, "scaler_cluster_final.pkl"))
joblib.dump(gmm, os.path.join(RESULTS_DIR, "gmm_regimes_final.pkl"))
print("All models and artifacts saved in", RESULTS_DIR)

# Final printout summary for quick copy-paste into paper
print("\n===== Quick Summary =====")
print(f"Train / Test split: {len(train_df)} / {len(test_df)} (total {len(df)})")
print(f"Global Test RMSE: {rmse:.3f}, MAE: {mae:.3f}, MAPE%: {mape:.3f}, sMAPE%: {smape_val:.3f}")
print(f"R2: {r2:.3f}, NSE: {nse_val:.3f}, Willmott d: {willmott:.3f}")
print(f"Total prediction time (s): {total_pred_time:.2f}, Avg per-sample (s): {avg_pred_time:.6f}")
print("Model sizes (MB):")
for k,v in model_sizes.items():
    print(f"  {k}: {v:.3f} MB")
print("=========================\n")

# ---------- End ----------
print("DMOFF pipeline completed (upgraded for reviewer concerns + extra metrics). Inspect results/ folder for outputs.")

In [ ]:
# ============================
# DMOFF — Full Jupyter Script (Reviewer-upgraded + Additional Metrics & Efficiency Logging)
# ============================
# Paste this whole cell in Jupyter and run.
# Make sure to change DATA_PATH to your local path.

# ---------- (Optional) Install required packages ----------
# Uncomment and run if packages are missing.
# !pip install numpy pandas matplotlib seaborn scikit-learn lightgbm optuna tensorflow odfpy pyexcel-ods3 shap joblib

# ---------- Imports ----------
import os
import warnings
warnings.filterwarnings("ignore")
import time
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import joblib
import shap
from datetime import timedelta
import psutil  # for memory if available
import pickle
import pathlib

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ---------- User-config ----------
DATA_PATH = r"E:\Abroad period research\Time-Series KD + Explainability\C3A9E300.ods"
TARGET_COL = "O3"
DATETIME_COL = "FECHA_HORA"
RESULTS_DIR = "results_dmoff_upgraded_with_metrics"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------- Helper functions ----------
def read_table(path):
    """Read .ods or .xlsx / .csv; try sensible engines."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".ods":
        try:
            df = pd.read_excel(path, engine="odf")
            return df
        except Exception as e:
            raise RuntimeError("Error reading .ods - install odfpy or convert to .xlsx. Exception: " + str(e))
    elif ext in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif ext == ".csv":
        return pd.read_csv(path)
    else:
        raise RuntimeError("Unsupported file extension: " + ext)

def create_time_features(df, datetime_col):
    df['hour'] = df[datetime_col].dt.hour
    df['day'] = df[datetime_col].dt.day
    df['weekday'] = df[datetime_col].dt.weekday
    df['month'] = df[datetime_col].dt.month
    # cyclic encodings
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['month_sin'] = np.sin(2*np.pi*(df['month']-1)/12)
    df['month_cos'] = np.cos(2*np.pi*(df['month']-1)/12)
    return df

def make_lags(df, cols, lags):
    for col in cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df

def train_test_chrono_split(df, test_size_hours):
    """Chronological split: last `test_size_hours` rows as test."""
    train = df.iloc[:-test_size_hours].copy()
    test = df.iloc[-test_size_hours:].copy()
    return train, test

# ---------- Additional metric helper functions (NEW, non-invasive) ----------
def smape(y_true, y_pred):
    """Symmetric mean absolute percentage error"""
    denom = (np.abs(y_true) + np.abs(y_pred))
    # avoid divide by zero
    mask = denom == 0
    denom[mask] = 1e-8
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def nse(y_true, y_pred):
    """Nash–Sutcliffe efficiency"""
    num = np.sum((y_true - y_pred)**2)
    denom = np.sum((y_true - np.mean(y_true))**2)
    if denom == 0:
        return np.nan
    return 1 - num/denom

def willmott_d(y_true, y_pred):
    """Willmott's index of agreement"""
    denom = np.sum((np.abs(y_pred - np.mean(y_true)) + np.abs(y_true - np.mean(y_true)))**2)
    num = np.sum((y_true - y_pred)**2)
    if denom == 0:
        return np.nan
    return 1 - num / denom

def model_file_size_mb(path):
    """Return file size in MB (if exists)"""
    try:
        return os.path.getsize(path) / (1024*1024)
    except Exception:
        return np.nan

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan

# ---------- Logging containers (NEW) ----------
metrics_log = {}   # will store runtime & size & performance per model/regime
train_timers = {}
model_sizes = {}
performance_summary = {}  # global metrics
per_regime_metrics = []

# ---------- 1) Load data ----------
print("Loading data from:", DATA_PATH)
df = read_table(DATA_PATH)
print("Raw columns:", df.columns.tolist())
df.columns = [c.strip() for c in df.columns]

# If DATETIME_COL missing, try to auto-detect
if DATETIME_COL not in df.columns:
    for c in df.columns:
        if "FECHA" in c.upper() or "HORA" in c.upper():
            DATETIME_COL = c
            print("Using datetime column:", DATETIME_COL)
            break

# Parse datetime
df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], dayfirst=True, errors='coerce')
if df[DATETIME_COL].isna().any():
    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL].astype(str), dayfirst=True, errors='coerce')
assert df[DATETIME_COL].notna().all(), "Some datetime values could not be parsed. Check your FECHA_HORA column."

df = df.sort_values(DATETIME_COL).reset_index(drop=True)

# rename simple columns if user uses longer names (common in their sample)
rename_map = {}
for col in df.columns:
    c = col.strip()
    if c.lower().startswith("dd"):
        rename_map[col] = "DD"
    if "NO2" in c.upper():
        rename_map[col] = "NO2"
    if "O3" == c.upper() or c.upper().endswith("-O3") or c.upper().startswith("O3"):
        rename_map[col] = "O3"
    if "PM10" in c.upper():
        rename_map[col] = "PM10"
    if "TMP" in c.upper() or "TEMP" in c.upper():
        rename_map[col] = "TMP"
    if "VV" in c.upper():
        rename_map[col] = "VV"
if rename_map:
    df = df.rename(columns=rename_map)
print("After renaming, columns:", df.columns.tolist())

assert TARGET_COL in df.columns, f"{TARGET_COL} not found in data columns."

# ---------- 2) Feature engineering ----------
df = create_time_features(df, DATETIME_COL)

# numeric columns used in clustering and modeling (only keep existing)
# model_cols = ["CO", "DD", "NO2", "PM10", "TMP", "VV"]
model_cols = ["DD", "NO2", "PM10", "TMP", "VV"]
model_cols = [c for c in model_cols if c in df.columns]

# create lags for O3 and main predictors
lags = [1,2,3,24]   # 1,2,3 hour and 24-hour lag
lags_for = [TARGET_COL] + model_cols
df = make_lags(df, lags_for, lags)
df = df.dropna().reset_index(drop=True)
print("Data shape after lags and dropna:", df.shape)

# ---------- 3) Discover regimes via clustering (GMM) - TRAIN-ONLY, with smoothing & validation ----------
cluster_features = [c for c in model_cols + ['hour_sin','hour_cos','month_sin','month_cos'] if c in df.columns]
print("Cluster features:", cluster_features)

# We'll split chronologically first to avoid leakage for the GMM selection
test_fraction = 0.2
n_test = int(len(df) * test_fraction)
train_df_all, test_df_all = train_test_chrono_split(df, test_size_hours=n_test)
print("Clustering will be fit on train only. Train rows:", len(train_df_all), "Test rows:", len(test_df_all))

# Fit scaler on train clustering features only
scaler_cluster = StandardScaler().fit(train_df_all[cluster_features].values)
Xc_train = scaler_cluster.transform(train_df_all[cluster_features].values)
Xc_test = scaler_cluster.transform(test_df_all[cluster_features].values)

# Choose number of clusters using BIC on train only
def select_gmm_k_on_train(X_train, k_min=2, k_max=6, random_state=RANDOM_SEED):
    best_bic = np.inf
    best_k = k_min
    best_model = None
    for k in range(k_min, k_max+1):
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=random_state, n_init=5)
        gmm.fit(X_train)
        bic = gmm.bic(X_train)
        print(f"GMM k={k} BIC={bic:.1f}")
        if bic < best_bic:
            best_bic = bic
            best_k = k
            best_model = gmm
    return best_k, best_model

print("Selecting number of regimes (GMM) on train only...")
best_k, gmm = select_gmm_k_on_train(Xc_train, k_min=2, k_max=5)
print("Selected regimes (k):", best_k)

# Assign regimes: predict on train & test using scaler and trained GMM
train_df_all['regime'] = gmm.predict(Xc_train)
test_df_all['regime'] = gmm.predict(Xc_test)

# Optional temporal smoothing to enforce regime persistence (small window)
SMOOTH_REGIMES = True
SMOOTH_WINDOW = 3  # odd integer
if SMOOTH_REGIMES:
    train_df_all['regime'] = train_df_all['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)
    test_df_all['regime'] = test_df_all['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)

# Compute clustering validity metrics on train
try:
    sil = silhouette_score(Xc_train, train_df_all['regime'])
    dbi = davies_bouldin_score(Xc_train, train_df_all['regime'])
    print(f"Clustering validity (train): Silhouette = {sil:.4f}, Davies-Bouldin = {dbi:.4f}")
    metrics_log['clustering'] = {'silhouette': float(sil), 'davies_bouldin': float(dbi)}
except Exception as e:
    print("Could not compute silhouette/DBI:", e)

# Create full df regimes by applying scaler and gmm predict to entire df (no refit)
Xc_full = scaler_cluster.transform(df[cluster_features].values)
df['regime'] = gmm.predict(Xc_full)
if SMOOTH_REGIMES:
    df['regime'] = df['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)

# Regime summary table (means)
regime_summary = df.groupby('regime')[cluster_features].mean().round(3)
print("Regime summary (means):")
display(regime_summary)

# plot regime counts
plt.figure(figsize=(6,3))
df['regime'].value_counts().sort_index().plot(kind='bar', title='Regime counts')
plt.show()

# Persist cluster scaler & gmm
joblib.dump(scaler_cluster, os.path.join(RESULTS_DIR, "scaler_cluster.pkl"))
joblib.dump(gmm, os.path.join(RESULTS_DIR, "gmm_regimes.pkl"))

# ---------- 4) Chronological train/test split for full modelling ----------
# We'll now use train_df_all (already computed) and test_df_all for training models
train_df = train_df_all.copy().reset_index(drop=True)
test_df = test_df_all.copy().reset_index(drop=True)
print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
metrics_log['data_split'] = {'n_total': int(len(df)), 'n_train': int(len(train_df)), 'n_test': int(len(test_df)),
                            'train_fraction': float(len(train_df)/len(df)), 'test_fraction': float(len(test_df)/len(df))}

# ---------- 5) Per-regime model training ----------
# Common feature set for tabular models
tab_features = []
# features include lags for each variable in lags_for (like O3_lag1...) plus meteorological/time features
for c in df.columns:
    # if column starts with any lag prefix or matches meta features
    if any([c.startswith(f"{col}_lag") for col in lags_for]) or c in model_cols + ['hour','weekday','month','hour_sin','hour_cos','month_sin','month_cos']:
        tab_features.append(c)
# ensure target not in features
tab_features = [c for c in tab_features if c != TARGET_COL and c != DATETIME_COL and c != 'regime']
print("Tabular features used (sample):", tab_features[:30])

# Setup storage
regime_models = {}
regimes_unique = sorted(train_df['regime'].unique())
print("Regimes to train:", regimes_unique)

# ---------- 5.A) Train Random Forest per regime WITH OPTUNA ----------
rf_total_train_time = 0.0
rf_regime_times = {}

def fit_rf_for_regime_optuna(Xtrain, ytrain, Xval, yval, n_trials=20, seed=RANDOM_SEED):
    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 50, 300)
        max_depth = trial.suggest_int("max_depth", 5, 50)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=seed,
            n_jobs=-1
        )
        
        rf.fit(Xtrain, ytrain)
        preds = rf.predict(Xval)
        return mean_squared_error(yval, preds, squared=False)  # RMSE

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    # Fit final model with best params on combined train+val
    best_params = study.best_params
    final_rf = RandomForestRegressor(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        max_features=best_params['max_features'],
        random_state=seed,
        n_jobs=-1
    )
    
    # Combine train and validation for final training
    X_combined = np.vstack([Xtrain, Xval])
    y_combined = np.concatenate([ytrain, yval])
    final_rf.fit(X_combined, y_combined)
    
    return final_rf, study

for r in regimes_unique:
    print("Training Random Forest for regime:", r)
    start_r = time.time()
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < 120:
        print(f"Regime {r} has too few samples ({len(df_r)}). Skipping per-regime RF.")
        continue
        
    split_idx = int(0.8 * len(df_r))
    train_r = df_r.iloc[:split_idx]
    val_r = df_r.iloc[split_idx:]
    
    Xtrain = train_r[tab_features].values
    ytrain = train_r[TARGET_COL].values
    Xval = val_r[tab_features].values
    yval = val_r[TARGET_COL].values
    
    scaler = StandardScaler().fit(np.vstack([Xtrain, Xval]))
    Xtrain_s = scaler.transform(Xtrain)
    Xval_s = scaler.transform(Xval)
    
    print("  Tuning Random Forest (Optuna)...")
    try:
        t0 = time.time()
        rf_model, study = fit_rf_for_regime_optuna(Xtrain_s, ytrain, Xval_s, yval, n_trials=20)
        t1 = time.time()
        elapsed = t1 - t0
        rf_total_train_time += elapsed
        rf_regime_times[r] = elapsed
        
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['rf_model'] = rf_model
        regime_models[r]['rf_scaler'] = scaler
        
        # Save model and record size
        rf_path = os.path.join(RESULTS_DIR, f"rf_regime_{r}.pkl")
        joblib.dump(rf_model, rf_path)
        size_mb = model_file_size_mb(rf_path)
        model_sizes[f"rf_regime_{r}"] = size_mb
        train_timers[f"rf_regime_{r}"] = elapsed
        
        print(f"  Random Forest done. Best RMSE (val): {study.best_value:.4f}, train time(s): {elapsed:.2f}, size(MB): {size_mb:.3f}")
    except Exception as e:
        print("  Random Forest training failed for regime", r, "due to:", e)

# Train global fallback Random Forest with Optuna
print("Training global fallback Random Forest on whole train set...")
rf_global = None
scaler_rf_glob = None
rf_global_train_time = None

try:
    t0_glob = time.time()
    split_idx = int(0.9 * len(train_df))
    train_glob = train_df.iloc[:split_idx]
    val_glob = train_df.iloc[split_idx:]
    
    Xtrain_g = train_glob[tab_features].values
    ytrain_g = train_glob[TARGET_COL].values
    Xval_g = val_glob[tab_features].values
    yval_g = val_glob[TARGET_COL].values
    
    scaler_rf_glob = StandardScaler().fit(np.vstack([Xtrain_g, Xval_g]))
    Xtrain_gs = scaler_rf_glob.transform(Xtrain_g)
    Xval_gs = scaler_rf_glob.transform(Xval_g)
    
    rf_global, study_glob = fit_rf_for_regime_optuna(Xtrain_gs, ytrain_g, Xval_gs, yval_g, n_trials=25)
    t1_glob = time.time()
    rf_global_train_time = t1_glob - t0_glob
    
    rf_global_path = os.path.join(RESULTS_DIR, "rf_global.pkl")
    joblib.dump(rf_global, rf_global_path)
    joblib.dump(scaler_rf_glob, os.path.join(RESULTS_DIR, "scaler_rf_global.pkl"))
    joblib.dump(study_glob, os.path.join(RESULTS_DIR, "optuna_rf_global.pkl"))
    
    size_glob = model_file_size_mb(rf_global_path)
    model_sizes["rf_global"] = size_glob
    train_timers["rf_global"] = rf_global_train_time
    
    print(f"Global RF trained. Best RMSE: {study_glob.best_value:.4f}, time(s): {rf_global_train_time:.2f}, size(MB): {size_glob:.3f}")
except Exception as e:
    print("Global Random Forest training failed:", e)

# ---------- 5.B) Train LightGBM per regime with Optuna ----------
def fit_lgb_for_regime(Xtrain, ytrain, Xval, yval, n_trials=25, seed=RANDOM_SEED):
    def objective(trial):
        param = {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "seed": seed,
            "num_leaves": trial.suggest_int("num_leaves", 16, 128),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        }

        dtrain = lgb.Dataset(Xtrain, label=ytrain)
        dvalid = lgb.Dataset(Xval, label=yval)

        bst = lgb.train(
            param,
            dtrain,
            valid_sets=[dvalid],
            num_boost_round=1000,
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(0)
            ]
        )

        preds = bst.predict(Xval, num_iteration=bst.best_iteration)
        return mean_squared_error(yval, preds, squared=False)  # RMSE

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    # Fit final model on combined train+val using best params
    best_params = study.best_params
    best_params.update({
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "seed": seed
    })

    dtrain_all = lgb.Dataset(np.vstack([Xtrain, Xval]), label=np.concatenate([ytrain, yval]))
    final = lgb.train(
        best_params,
        dtrain_all,
        num_boost_round=200  # keep reasonable
    )

    # store iteration info
    study.best_trial.user_attrs["best_iteration"] = final.current_iteration() if hasattr(final, "current_iteration") else 200
    return final, study

# Timing variables for LGB training
lgb_total_train_time = 0.0
lgb_regime_times = {}

for r in regimes_unique:
    print("Training LGB for regime:", r)
    start_r = time.time()
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < 120:
        print(f"Regime {r} has too few samples ({len(df_r)}). Skipping per-regime LGB (will use global fallback later).")
        continue
    split_idx = int(0.8 * len(df_r))
    train_r = df_r.iloc[:split_idx]
    val_r = df_r.iloc[split_idx:]
    Xtrain = train_r[tab_features].values
    ytrain = train_r[TARGET_COL].values
    Xval = val_r[tab_features].values
    yval = val_r[TARGET_COL].values
    scaler = StandardScaler().fit(np.vstack([Xtrain, Xval]))
    Xtrain_s = scaler.transform(Xtrain)
    Xval_s = scaler.transform(Xval)
    print("  Tuning LightGBM (Optuna)...")
    try:
        t0 = time.time()
        lgb_model, study = fit_lgb_for_regime(Xtrain_s, ytrain, Xval_s, yval, n_trials=25)
        t1 = time.time()
        elapsed = t1 - t0
        lgb_total_train_time += elapsed
        lgb_regime_times[r] = elapsed
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['lgb_model'] = lgb_model
        regime_models[r]['lgb_scaler'] = scaler
        joblib.dump(study, os.path.join(RESULTS_DIR, f"optuna_lgb_regime_{r}.pkl"))
        # Save model and record size
        lgb_path = os.path.join(RESULTS_DIR, f"lgb_regime_{r}.txt")
        lgb_model.save_model(lgb_path)
        size_mb = model_file_size_mb(lgb_path)
        model_sizes[f"lgb_regime_{r}"] = size_mb
        train_timers[f"lgb_regime_{r}"] = elapsed
        print("  LightGBM done. best RMSE (val):", study.best_value, "train time(s):", round(elapsed,2), "size(MB):", round(size_mb,3))
    except Exception as e:
        print("  LightGBM training failed for regime", r, "due to:", e)

# Train a global fallback LightGBM on all train data for regimes with too few samples
print("Training global fallback LightGBM on whole train set (for small regimes)...")
lgb_global = None
scaler_glob = None
lgb_global_train_time = None
try:
    t0_glob = time.time()
    split_idx = int(0.9 * len(train_df))
    train_glob = train_df.iloc[:split_idx]
    val_glob = train_df.iloc[split_idx:]
    Xtrain_g = train_glob[tab_features].values
    ytrain_g = train_glob[TARGET_COL].values
    Xval_g = val_glob[tab_features].values
    yval_g = val_glob[TARGET_COL].values
    scaler_glob = StandardScaler().fit(np.vstack([Xtrain_g, Xval_g]))
    Xtrain_gs = scaler_glob.transform(Xtrain_g)
    Xval_gs = scaler_glob.transform(Xval_g)
    lgb_global, study_glob = fit_lgb_for_regime(Xtrain_gs, ytrain_g, Xval_gs, yval_g, n_trials=20)
    t1_glob = time.time()
    lgb_global_train_time = t1_glob - t0_glob
    joblib.dump(lgb_global, os.path.join(RESULTS_DIR, "lgb_global.pkl"))
    joblib.dump(scaler_glob, os.path.join(RESULTS_DIR, "scaler_lgb_global.pkl"))
    # save text model
    lgb_global_path = os.path.join(RESULTS_DIR, "lgb_global.txt")
    lgb_global.save_model(lgb_global_path)
    size_glob = model_file_size_mb(lgb_global_path)
    model_sizes["lgb_global"] = size_glob
    train_timers["lgb_global"] = lgb_global_train_time
    print("Global LGB trained. best RMSE:", study_glob.best_value, "time(s):", round(lgb_global_train_time,2), "size(MB):", round(size_glob,3))
except Exception as e:
    print("Global LightGBM training failed:", e)

# ---------- 5.C) Train small LSTM per regime ----------
SEQ_LEN = 24  # use last 24 hours to predict next hour

def make_sequences(df_reg, seq_len, feature_cols, target_col):
    Xs, ys = [], []
    arr = df_reg[feature_cols].values
    tar = df_reg[target_col].values
    for i in range(seq_len, len(df_reg)):
        Xs.append(arr[i-seq_len:i])
        ys.append(tar[i])
    return np.array(Xs), np.array(ys)

lstm_histories = {}
lstm_train_times = {}
lstm_total_time = 0.0

for r in regimes_unique:
    print("Training LSTM for regime:", r)
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < SEQ_LEN + 30:
        print(f"  Too few samples ({len(df_r)}). Skipping LSTM for regime {r}.")
        continue
    seq_features = [c for c in tab_features]  # use same tab features in sequence
    scaler_seq = MinMaxScaler()
    scaler_seq.fit(df_r[seq_features].values)
    arr_scaled = scaler_seq.transform(df_r[seq_features].values)
    df_r_scaled = df_r.copy()
    df_r_scaled[seq_features] = arr_scaled
    Xs, ys = make_sequences(df_r_scaled, SEQ_LEN, seq_features, TARGET_COL)
    split = int(0.8 * len(Xs))
    Xtrain, Xval = Xs[:split], Xs[split:]
    ytrain, yval = ys[:split], ys[split:]
    def build_lstm_model(n_features, lr=1e-3, units=64, dropout=0.2):
        inp = keras.Input(shape=(SEQ_LEN, n_features))
        x = layers.LSTM(units, return_sequences=False)(inp)
        x = layers.Dropout(dropout)(x)
        out = layers.Dense(1)(x)
        model = keras.Model(inputs=inp, outputs=out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss='mse')
        return model
    def optuna_lstm_objective(trial):
        units = trial.suggest_int("units", 16, 128)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
        model = build_lstm_model(Xtrain.shape[2], lr=lr, units=units, dropout=dropout)
        early = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')
        h = model.fit(Xtrain, ytrain, validation_data=(Xval, yval), epochs=60, batch_size=32, callbacks=[early], verbose=0)
        val_rmse = np.sqrt(min(h.history['val_loss']))
        return val_rmse
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    n_trials = 20
    try:
        t0 = time.time()
        study.optimize(optuna_lstm_objective, n_trials=n_trials, show_progress_bar=False)
        t1 = time.time()
        elapsed = t1 - t0
        lstm_total_time += elapsed
        lstm_train_times[r] = elapsed
        print("  Best LSTM RMSE (val):", study.best_value, "t(s):", round(elapsed,2))
        best = study.best_params
        final_model = build_lstm_model(Xs.shape[2], lr=best.get('lr',1e-3), units=best.get('units',64), dropout=best.get('dropout',0.2))
        early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='loss')
        t0f = time.time()
        final_model.fit(Xs, ys, epochs=100, batch_size=32, callbacks=[early], verbose=0)
        t1f = time.time()
        fit_elapsed = t1f - t0f
        total_elapsed = elapsed + fit_elapsed
        lstm_train_times[r] = total_elapsed
        lstm_histories[r] = study.best_value
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['lstm_model'] = final_model
        regime_models[r]['lstm_scaler'] = scaler_seq
        # save model (h5) and size
        lstm_path = os.path.join(RESULTS_DIR, f"lstm_regime_{r}.h5")
        try:
            final_model.save(lstm_path)
        except Exception:
            # fallback to pickle for keras (may not be ideal)
            with open(os.path.join(RESULTS_DIR, f"lstm_regime_{r}.pkl"), "wb") as f:
                pickle.dump(final_model, f)
            lstm_path = os.path.join(RESULTS_DIR, f"lstm_regime_{r}.pkl")
        size_mb = model_file_size_mb(lstm_path)
        model_sizes[f"lstm_regime_{r}"] = size_mb
        train_timers[f"lstm_regime_{r}"] = total_elapsed
        joblib.dump(study, os.path.join(RESULTS_DIR, f"optuna_lstm_regime_{r}.pkl"))
        print(f"  Final LSTM fit time(s): {round(fit_elapsed,2)}, total(s): {round(total_elapsed,2)}, size(MB): {round(size_mb,3)}")
    except Exception as e:
        print("  LSTM training failed for regime", r, "due to:", e)

# ---------- 6) Train regime classifier (meta-controller) ----------
# The classifier predicts the regime at time t given current features (trained on train set only)
clf_features = cluster_features + [TARGET_COL] + [f"{TARGET_COL}_lag{l}" for l in lags]
clf_features = [c for c in clf_features if c in train_df.columns]
print("Regime classifier features:", clf_features)

Xclf = train_df[clf_features].values
yclf = train_df['regime'].values
clf_scaler = StandardScaler().fit(Xclf)
Xclf_s = clf_scaler.transform(Xclf)
t0_clf = time.time()
rfc = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED)
rfc.fit(Xclf_s, yclf)
t1_clf = time.time()
clf_train_time = t1_clf - t0_clf
joblib.dump(rfc, os.path.join(RESULTS_DIR, "regime_classifier.pkl"))
joblib.dump(clf_scaler, os.path.join(RESULTS_DIR, "regime_clf_scaler.pkl"))
# record size/time
rfc_path = os.path.join(RESULTS_DIR, "regime_classifier.pkl")
model_sizes["regime_classifier_rf"] = model_file_size_mb(rfc_path)
train_timers["regime_classifier_rf"] = clf_train_time
print("Regime classifier trained. Train accuracy (on scaled train):", rfc.score(Xclf_s, yclf), "time(s):", round(clf_train_time,2), "size(MB):", round(model_sizes["regime_classifier_rf"],3))

# ---------- 7) Attention-style meta-blender ----------
from tensorflow.keras import backend as K

def build_blender(input_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    w = layers.Dense(2, activation='softmax')(x)  # outputs two weights summing to 1
    model = keras.Model(inputs=inp, outputs=w)
    # we'll use custom training loop below
    return model

# Prepare blender training data: use last 10% of train_df (chronological)
blend_df = train_df.iloc[int(0.9*len(train_df)):].copy().reset_index(drop=True)
blend_features = tab_features  # use tab features as input to blender
X_blend = []
Y_blend = []
preds_lgb_all = []
preds_lstm_all = []

# We'll use the global fallback LGB/scaler if per-regime LGB missing
lgb_global_exists = os.path.exists(os.path.join(RESULTS_DIR, "lgb_global.pkl"))
if lgb_global_exists:
    try:
        lgb_global = joblib.load(os.path.join(RESULTS_DIR, "lgb_global.pkl"))
        scaler_glob = joblib.load(os.path.join(RESULTS_DIR, "scaler_lgb_global.pkl"))
    except:
        lgb_global = None
        scaler_glob = None
else:
    lgb_global = None
    scaler_glob = None

# Check if RF global exists
rf_global_exists = os.path.exists(os.path.join(RESULTS_DIR, "rf_global.pkl"))
if rf_global_exists:
    try:
        rf_global = joblib.load(os.path.join(RESULTS_DIR, "rf_global.pkl"))
        scaler_rf_glob = joblib.load(os.path.join(RESULTS_DIR, "scaler_rf_global.pkl"))
    except:
        rf_global = None
        scaler_rf_glob = None
else:
    rf_global = None
    scaler_rf_glob = None

for idx, row in blend_df.reset_index(drop=True).iterrows():
    r = int(row['regime'])
    # LGB prediction
    if 'lgb_model' in regime_models.get(r, {}):
        scaler_lgb = regime_models[r]['lgb_scaler']
        lgb_model = regime_models[r]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_global.predict(X_row)[0])
    else:
        pred_lgb = float(row[TARGET_COL])  # fallback to persistence

    # LSTM prediction: build sequence from training data only (NO DATA LEAKAGE)
    if 'lstm_model' in regime_models.get(r, {}):
        lstm_model = regime_models[r]['lstm_model']
        scaler_seq = regime_models[r]['lstm_scaler']
        dt = row[DATETIME_COL]
        # Use training data only for LSTM sequences - NO DATA LEAKAGE
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==r)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values)
            seq = seq.reshape(1, SEQ_LEN, len(tab_features))
            try:
                pred_lstm = float(lstm_model.predict(seq)[0,0])
            except:
                pred_lstm = pred_lgb
        else:
            pred_lstm = pred_lgb
    else:
        pred_lstm = pred_lgb

    preds_lgb_all.append(pred_lgb)
    preds_lstm_all.append(pred_lstm)
    X_blend.append(row[blend_features].values)
    Y_blend.append(row[TARGET_COL])

X_blend = np.array(X_blend)
Y_blend = np.array(Y_blend)
preds_lgb_all = np.array(preds_lgb_all)
preds_lstm_all = np.array(preds_lstm_all)

scaler_blend = StandardScaler().fit(X_blend)
X_blend_scaled = scaler_blend.transform(X_blend)
X_blend_in = np.hstack([X_blend_scaled, preds_lgb_all.reshape(-1,1), preds_lstm_all.reshape(-1,1)])
blender_input_dim = X_blend_in.shape[1]
print("Blender input dim:", blender_input_dim)

def blender_train_end_to_end(Xin, predsA, predsB, Y, batch=32, epochs=60):
    # Build model that outputs weights; we'll train with a custom loop minimizing MSE between weighted preds and Y
    inp = keras.Input(shape=(Xin.shape[1],))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    w = layers.Dense(2, activation='softmax')(x)
    model = keras.Model(inputs=inp, outputs=w)
    optimizer = keras.optimizers.Adam()
    # prepare dataset
    N = len(Xin)
    for epoch in range(epochs):
        idxs = np.arange(N)
        np.random.shuffle(idxs)
        epoch_losses = []
        for i in range(0, N, batch):
            batch_idx = idxs[i:i+batch]
            Xin_b = Xin[batch_idx]
            predsA_b = predsA[batch_idx]
            predsB_b = predsB[batch_idx]
            Y_b = Y[batch_idx]
            with tf.GradientTape() as tape:
                W_b = model(Xin_b, training=True)  # shape (b,2)
                combined = W_b[:,0]*predsA_b + W_b[:,1]*predsB_b
                loss = tf.reduce_mean(tf.square(combined - Y_b))
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            epoch_losses.append(float(loss))
        if epoch % 10 == 0 or epoch == epochs-1:
            print(f"Blender epoch {epoch}, loss {np.mean(epoch_losses):.6f}")
    return model

print("Training blender network (end-to-end)...")
t0_blend = time.time()
blender_model = blender_train_end_to_end(X_blend_in, preds_lgb_all, preds_lstm_all, Y_blend, batch=32, epochs=60)
t1_blend = time.time()
blender_train_time = t1_blend - t0_blend
# save blender & scaler
joblib.dump(scaler_blend, os.path.join(RESULTS_DIR, "scaler_blend.pkl"))
blender_path = os.path.join(RESULTS_DIR, "blender_model.h5")
try:
    blender_model.save(blender_path)
except Exception:
    # fallback to pickle
    with open(os.path.join(RESULTS_DIR, "blender_model.pkl"), "wb") as f:
        pickle.dump(blender_model, f)
    blender_path = os.path.join(RESULTS_DIR, "blender_model.pkl")
model_sizes["blender_model"] = model_file_size_mb(blender_path)
train_timers["blender_model"] = blender_train_time
print("Blender trained. time(s):", round(blender_train_time,2), "size(MB):", round(model_sizes["blender_model"],3))

# ---------- 8) Prediction pipeline on test set ----------
def predict_one_row(row):
    """Given a row from test_df, predict O3 for next hour using DMOFF pipeline."""
    # predict regime for current row using regime classifier (trained on train only)
    xf = row[clf_features].values.reshape(1,-1)
    xf_s = clf_scaler.transform(xf)
    pred_reg = int(rfc.predict(xf_s)[0])

    # LGB prediction (per-regime or global fallback)
    if 'lgb_model' in regime_models.get(pred_reg, {}):
        scaler_lgb = regime_models[pred_reg]['lgb_scaler']
        lgb_model = regime_models[pred_reg]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_global.predict(X_row)[0])
    else:
        pred_lgb = float(row[TARGET_COL])

    # LSTM prediction - USE TRAINING DATA ONLY (NO DATA LEAKAGE)
    if 'lstm_model' in regime_models.get(pred_reg, {}):
        lstm_model = regime_models[pred_reg]['lstm_model']
        scaler_seq = regime_models[pred_reg]['lstm_scaler']
        dt = row[DATETIME_COL]
        # Use training data only for LSTM sequences - NO DATA LEAKAGE
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==pred_reg)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values).reshape(1, SEQ_LEN, len(tab_features))
            try:
                pred_lstm = float(lstm_model.predict(seq)[0,0])
            except:
                pred_lstm = pred_lgb
        else:
            pred_lstm = pred_lgb
    else:
        pred_lstm = pred_lgb

    # Build blender input
    Xb_raw = row[blend_features].values.reshape(1,-1)
    Xb_scaled = scaler_blend.transform(Xb_raw)
    Xb_in = np.hstack([Xb_scaled, np.array([[pred_lgb, pred_lstm]])])
    w = blender_model.predict(Xb_in)[0]  # two weights
    pred = w[0]*pred_lgb + w[1]*pred_lstm
    return pred, pred_reg, pred_lgb, pred_lstm, w

# Run predictions across test set with timing
y_true = []
y_pred = []
pred_details = []
pred_times = []
start_all_preds = time.time()
for idx, row in test_df.iterrows():
    t0p = time.time()
    p, preg, plgb, plstm, w = predict_one_row(row)
    t1p = time.time()
    pred_times.append(t1p - t0p)
    y_true.append(row[TARGET_COL])
    y_pred.append(p)
    pred_details.append((preg, plgb, plstm, w))
end_all_preds = time.time()
total_pred_time = end_all_preds - start_all_preds
avg_pred_time = np.mean(pred_times) if pred_times else np.nan
pred_percentiles = np.percentile(pred_times, [50, 90, 95]) if pred_times else [np.nan, np.nan, np.nan]
metrics_log['prediction_time'] = {'total_s': float(total_pred_time), 'avg_per_sample_s': float(avg_pred_time),
                                 'p50_s': float(pred_percentiles[0]), 'p90_s': float(pred_percentiles[1]), 'p95_s': float(pred_percentiles[2])}

y_true = np.array(y_true); y_pred = np.array(y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
mape = None
try:
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100.0
except Exception:
    # fallback MAPE
    mape = np.mean(np.abs((y_true - y_pred) / (np.where(y_true == 0, 1e-8, y_true)))) * 100.0
smape_val = smape(y_true, y_pred)
r2 = safe_r2(y_true, y_pred)
nse_val = nse(y_true, y_pred)
willmott = willmott_d(y_true, y_pred)
median_ae = median_absolute_error(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}, MAE: {mae:.3f}, MAPE%: {mape:.3f}, sMAPE%: {smape_val:.3f}")
print(f"R2: {r2:.3f}, NSE: {nse_val:.3f}, Willmott d: {willmott:.3f}, MedianAE: {median_ae:.3f}")

# Save predictions
test_df_out = test_df.copy()
test_df_out['pred_dmoff'] = y_pred
test_df_out['obs'] = y_true
test_df_out.to_csv(os.path.join(RESULTS_DIR, "test_predictions_dmoff.csv"), index=False)

# ---------- 9) COMPREHENSIVE MODEL COMPARISON (INCLUDING ALL BASE MODELS) ----------
print("\n" + "="*80)
print("COMPREHENSIVE MODEL PERFORMANCE COMPARISON")
print("="*80)

# Store individual model predictions
test_df_out['pred_rf_only'] = np.nan
test_df_out['pred_lgb_only'] = np.nan
test_df_out['pred_lstm_only'] = np.nan
test_df_out['pred_regime'] = np.nan

# Get individual model predictions for ALL models
for idx, row in test_df_out.iterrows():
    r = int(row['regime'])
    
    # Random Forest prediction
    if 'rf_model' in regime_models.get(r, {}):
        scaler_rf = regime_models[r]['rf_scaler']
        rf_model = regime_models[r]['rf_model']
        X_row = scaler_rf.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_rf_only'] = float(rf_model.predict(X_row)[0])
    elif rf_global is not None:
        X_row = scaler_rf_glob.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_rf_only'] = float(rf_global.predict(X_row)[0])
    
    # LightGBM only prediction
    if 'lgb_model' in regime_models.get(r, {}):
        scaler_lgb = regime_models[r]['lgb_scaler']
        lgb_model = regime_models[r]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_lgb_only'] = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_lgb_only'] = float(lgb_global.predict(X_row)[0])
    
    # LSTM only prediction (with the data leakage fix!)
    if 'lstm_model' in regime_models.get(r, {}):
        lstm_model = regime_models[r]['lstm_model']
        scaler_seq = regime_models[r]['lstm_scaler']
        dt = row[DATETIME_COL]
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==r)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values).reshape(1, SEQ_LEN, len(tab_features))
            try:
                test_df_out.at[idx, 'pred_lstm_only'] = float(lstm_model.predict(seq)[0,0])
            except:
                test_df_out.at[idx, 'pred_lstm_only'] = test_df_out.at[idx, 'pred_lgb_only']
    
    # Store the predicted regime
    test_df_out.at[idx, 'pred_regime'] = r

# Calculate comprehensive metrics for each model
models_comparison = []

# Global performance for each model
for model_name, pred_col in [('DMOFF (Hybrid)', 'pred_dmoff'), 
                            ('Random Forest', 'pred_rf_only'), 
                            ('LightGBM Only', 'pred_lgb_only'), 
                            ('LSTM Only', 'pred_lstm_only')]:
    
    if pred_col not in test_df_out.columns or test_df_out[pred_col].isna().all():
        continue
        
    mask = test_df_out[pred_col].notna()
    y_true_model = test_df_out.loc[mask, 'obs'].values
    y_pred_model = test_df_out.loc[mask, pred_col].values
    
    if len(y_true_model) > 0:
        rmse_m = np.sqrt(mean_squared_error(y_true_model, y_pred_model))
        mae_m = mean_absolute_error(y_true_model, y_pred_model)
        r2_m = safe_r2(y_true_model, y_pred_model)
        smape_m = smape(y_true_model, y_pred_model)
        mape_m = None
        try:
            mape_m = mean_absolute_percentage_error(y_true_model, y_pred_model) * 100.0
        except Exception:
            mape_m = np.mean(np.abs((y_true_model - y_pred_model) / (np.where(y_true_model == 0, 1e-8, y_true_model)))) * 100.0
        nse_m = nse(y_true_model, y_pred_model)
        willmott_m = willmott_d(y_true_model, y_pred_model)
        median_ae_m = median_absolute_error(y_true_model, y_pred_model)
        
        models_comparison.append({
            'model': model_name,
            'n_samples': len(y_true_model),
            'rmse': rmse_m,
            'mae': mae_m,
            'r2': r2_m,
            'mape_pct': mape_m,
            'smape_pct': smape_m,
            'nse': nse_m,
            'willmott_d': willmott_m,
            'median_ae': median_ae_m
        })
        
        print(f"{model_name:20} | RMSE: {rmse_m:.3f} | MAE: {mae_m:.3f} | R²: {r2_m:.3f} | sMAPE: {smape_m:.3f}%")

# Create a comprehensive comparison table
print("\n" + "="*100)
print("DETAILED MODEL COMPARISON TABLE")
print("="*100)
comparison_df = pd.DataFrame(models_comparison)
if not comparison_df.empty:
    display(comparison_df.round(4))

# Per-regime performance for each model
print("\n" + "="*80)
print("PER-REGIME PERFORMANCE COMPARISON")
print("="*80)
for r in sorted(test_df_out['regime'].unique()):
    df_r = test_df_out[test_df_out['regime']==r]
    if len(df_r) == 0:
        continue
        
    print(f"\nRegime {r} (N={len(df_r)}):")
    print("-" * 50)
    
    regime_comparison = []
    for model_name, pred_col in [('DMOFF', 'pred_dmoff'), 
                                ('Random Forest', 'pred_rf_only'), 
                                ('LightGBM', 'pred_lgb_only'), 
                                ('LSTM', 'pred_lstm_only')]:
        
        if pred_col not in df_r.columns or df_r[pred_col].isna().all():
            continue
            
        mask = df_r[pred_col].notna()
        if mask.sum() == 0:
            continue
            
        yt = df_r.loc[mask, 'obs'].values
        yp = df_r.loc[mask, pred_col].values
        
        rmse_r = np.sqrt(mean_squared_error(yt, yp))
        mae_r = mean_absolute_error(yt, yp)
        r2_r = safe_r2(yt, yp)
        
        regime_comparison.append({
            'model': model_name,
            'rmse': rmse_r,
            'mae': mae_r,
            'r2': r2_r
        })
        
        print(f"  {model_name:15} | RMSE: {rmse_r:.3f} | MAE: {mae_r:.3f} | R²: {r2_r:.3f}")
    
    # Find best model for this regime
    if regime_comparison:
        best_rmse = min([m['rmse'] for m in regime_comparison])
        best_models = [m['model'] for m in regime_comparison if m['rmse'] == best_rmse]
        print(f"  Best in regime {r}: {', '.join(best_models)} (RMSE: {best_rmse:.3f})")

# Save comprehensive comparison results
comparison_df = pd.DataFrame(models_comparison)
comparison_df.to_csv(os.path.join(RESULTS_DIR, "comprehensive_model_comparison_metrics.csv"), index=False)
test_df_out.to_csv(os.path.join(RESULTS_DIR, "test_predictions_all_models.csv"), index=False)

print(f"\nComprehensive comparison results saved to: comprehensive_model_comparison_metrics.csv")

# # ---------- 10) Visualization of All Models Comparison ----------
# plt.figure(figsize=(15, 10))

# # Plot 1: Time series comparison
# plt.subplot(2, 2, 1)
# plt.plot(test_df_out[DATETIME_COL], test_df_out['obs'], label='Observed', color='black', linewidth=2, alpha=0.8)
# plt.plot(test_df_out[DATETIME_COL], test_df_out['pred_dmoff'], label='DMOFF (Hybrid)', alpha=0.8)
# plt.plot(test_df_out[DATETIME_COL], test_df_out['pred_rf_only'], label='Random Forest', alpha=0.7)
# plt.plot(test_df_out[DATETIME_COL], test_df_out['pred_lgb_only'], label='LightGBM', alpha=0.7)
# plt.plot(test_df_out[DATETIME_COL], test_df_out['pred_lstm_only'], label='LSTM', alpha=0.7)
# plt.legend()
# plt.title('Observed vs Predicted Values (All Models)')
# plt.xticks(rotation=45)

# # Plot 2: Scatter plots for each model
# models_to_plot = [('DMOFF', 'pred_dmoff'), ('Random Forest', 'pred_rf_only'), 
#                   ('LightGBM', 'pred_lgb_only'), ('LSTM', 'pred_lstm_only')]

# for i, (model_name, pred_col) in enumerate(models_to_plot):
#     if pred_col in test_df_out.columns:
#         plt.subplot(2, 2, i+2)
#         mask = test_df_out[pred_col].notna()
#         if mask.any():
#             y_true_plot = test_df_out.loc[mask, 'obs']
#             y_pred_plot = test_df_out.loc[mask, pred_col]
#             plt.scatter(y_true_plot, y_pred_plot, alpha=0.6, s=20)
#             plt.plot([y_true_plot.min(), y_true_plot.max()], [y_true_plot.min(), y_true_plot.max()], 'r--', alpha=0.8)
#             plt.xlabel('Observed')
#             plt.ylabel('Predicted')
#             rmse_plot = np.sqrt(mean_squared_error(y_true_plot, y_pred_plot))
#             r2_plot = safe_r2(y_true_plot, y_pred_plot)
#             plt.title(f'{model_name}\nRMSE: {rmse_plot:.3f}, R²: {r2_plot:.3f}')

# plt.tight_layout()
# plt.show()

# # ---------- 11) Performance Summary Bar Chart ----------
# plt.figure(figsize=(12, 8))

# # Create normalized performance metrics for visualization
# metrics_for_plot = ['rmse', 'mae', 'smape_pct']
# normalized_scores = {}

# for metric in metrics_for_plot:
#     if metric in comparison_df.columns:
#         values = comparison_df[metric].values
#         # For RMSE, MAE, sMAPE - lower is better, so we invert for the plot
#         if metric in ['rmse', 'mae', 'smape_pct']:
#             normalized = 1 - (values - values.min()) / (values.max() - values.min() + 1e-8)
#         else:
#             normalized = (values - values.min()) / (values.max() - values.min() + 1e-8)
#         normalized_scores[metric] = normalized

# Create bar chart
# x_pos = np.arange(len(comparison_df))
# bar_width = 0.25

# for i, metric in enumerate(metrics_for_plot):
#     if metric in normalized_scores:
#         plt.bar(x_pos + i * bar_width, normalized_scores[metric], bar_width, 
#                 label=metric.upper(), alpha=0.8)

# plt.xlabel('Models')
# plt.ylabel('Normalized Performance (Higher is Better)')
# plt.title('Model Performance Comparison (Normalized Metrics)')
# plt.xticks(x_pos + bar_width, comparison_df['model'], rotation=45)
# plt.legend()
# plt.tight_layout()
# plt.show()

# ---------- 12) Efficiency & performance summary ----------
# Collect global performance metrics
performance_summary = {
    'rmse': float(rmse),
    'mae': float(mae),
    'mape_pct': float(mape),
    'smape_pct': float(smape_val),
    'r2': float(r2) if not np.isnan(r2) else None,
    'nse': float(nse_val) if not np.isnan(nse_val) else None,
    'willmott_d': float(willmott) if not np.isnan(willmott) else None,
    'median_ae': float(median_ae),
    'n_test_samples': int(len(test_df)),
    'total_prediction_time_s': float(total_pred_time),
    'avg_prediction_time_per_sample_s': float(avg_pred_time)
}

# Merge with metrics_log
metrics_log['performance'] = performance_summary
metrics_log['model_sizes_mb'] = model_sizes
metrics_log['train_times_s'] = train_timers
metrics_log['rf_per_regime_train_times'] = {str(k): float(v) for k,v in rf_regime_times.items()}
metrics_log['lgb_per_regime_train_times'] = {str(k): float(v) for k,v in lgb_regime_times.items()}
metrics_log['lstm_per_regime_train_times'] = {str(k): float(v) for k,v in lstm_train_times.items()}
metrics_log['blender_train_time_s'] = float(blender_train_time)

# Save metrics_log as JSON
with open(os.path.join(RESULTS_DIR, "metrics_log.json"), "w") as f:
    json.dump(metrics_log, f, indent=2)

# Write a consolidated CSV with global and summary info
global_metrics_df = pd.DataFrame([performance_summary])
global_metrics_df.to_csv(os.path.join(RESULTS_DIR, "global_performance_metrics.csv"), index=False)

# Save comprehensive comparison
report = {
    'global_metrics': performance_summary,
    'model_comparison': models_comparison,
    'model_sizes_mb': model_sizes,
    'train_times_s': train_timers,
    'clustering_metrics': metrics_log.get('clustering', {})
}
with open(os.path.join(RESULTS_DIR, "comprehensive_dmoff_report.json"), "w") as f:
    json.dump(report, f, indent=2)

# ---------- 13) Save everything ----------
joblib.dump(regime_models, os.path.join(RESULTS_DIR, "regime_models.pkl"))
joblib.dump(rfc, os.path.join(RESULTS_DIR, "regime_classifier_rf.pkl"))
joblib.dump(scaler_cluster, os.path.join(RESULTS_DIR, "scaler_cluster_final.pkl"))
joblib.dump(gmm, os.path.join(RESULTS_DIR, "gmm_regimes_final.pkl"))
print("All models and artifacts saved in", RESULTS_DIR)

# Final comprehensive summary
print("\n" + "="*80)
print("COMPREHENSIVE SUMMARY - ALL MODELS")
print("="*80)
print(f"Dataset: {len(train_df)} train, {len(test_df)} test samples")
print(f"Regimes identified: {best_k}")
print("\nOverall Performance Ranking:")
print("-" * 50)

# Sort models by RMSE (ascending)
sorted_models = sorted(models_comparison, key=lambda x: x['rmse'])
for i, model in enumerate(sorted_models, 1):
    print(f"{i}. {model['model']:20} | RMSE: {model['rmse']:.3f} | MAE: {model['mae']:.3f} | R²: {model['r2']:.3f}")

print(f"\nDMOFF Hybrid Improvement over Best Base Model:")
if len(sorted_models) > 1:
    best_base = sorted_models[1]  # First after DMOFF (assuming DMOFF is best)
    dmoff = sorted_models[0]
    rmse_improvement = ((best_base['rmse'] - dmoff['rmse']) / best_base['rmse']) * 100
    mae_improvement = ((best_base['mae'] - dmoff['mae']) / best_base['mae']) * 100
    print(f"RMSE Improvement: {rmse_improvement:.1f}% over {best_base['model']}")
    print(f"MAE Improvement: {mae_improvement:.1f}% over {best_base['model']}")

print(f"\nTotal prediction time: {total_pred_time:.2f}s")
print(f"Average prediction time per sample: {avg_pred_time:.6f}s")
print("="*80)

# ---------- End ----------
print("\nDMOFF pipeline completed with comprehensive model comparison.")
print("All results saved to:", RESULTS_DIR)
print("Check 'comprehensive_model_comparison_metrics.csv' for detailed comparison.")

Consistent forecasting pefromance metrics across global and regime levels

In [ ]:
# ============================
# DMOFF — Full Jupyter Script with Consistent Metrics
# ============================
# Paste this whole cell in Jupyter and run.
# Make sure to change DATA_PATH to your local path.

# ---------- (Optional) Install required packages ----------
# Uncomment and run if packages are missing.
# !pip install numpy pandas matplotlib seaborn scikit-learn lightgbm optuna tensorflow odfpy pyexcel-ods3 shap joblib

# ---------- Imports ----------
import os
import warnings
warnings.filterwarnings("ignore")
import time
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import joblib
import shap
from datetime import timedelta
import psutil  # for memory if available
import pickle
import pathlib

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ---------- User-config ----------
DATA_PATH = r"E:\Abroad period research\Time-Series KD + Explainability\C3A9E300.ods"
TARGET_COL = "O3"
DATETIME_COL = "FECHA_HORA"
RESULTS_DIR = "results_dmoff_consistent_metrics"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------- Helper functions ----------
def read_table(path):
    """Read .ods or .xlsx / .csv; try sensible engines."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".ods":
        try:
            df = pd.read_excel(path, engine="odf")
            return df
        except Exception as e:
            raise RuntimeError("Error reading .ods - install odfpy or convert to .xlsx. Exception: " + str(e))
    elif ext in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif ext == ".csv":
        return pd.read_csv(path)
    else:
        raise RuntimeError("Unsupported file extension: " + ext)

def create_time_features(df, datetime_col):
    df['hour'] = df[datetime_col].dt.hour
    df['day'] = df[datetime_col].dt.day
    df['weekday'] = df[datetime_col].dt.weekday
    df['month'] = df[datetime_col].dt.month
    # cyclic encodings
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['month_sin'] = np.sin(2*np.pi*(df['month']-1)/12)
    df['month_cos'] = np.cos(2*np.pi*(df['month']-1)/12)
    return df

def make_lags(df, cols, lags):
    for col in cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df

def train_test_chrono_split(df, test_size_hours):
    """Chronological split: last `test_size_hours` rows as test."""
    train = df.iloc[:-test_size_hours].copy()
    test = df.iloc[-test_size_hours:].copy()
    return train, test

# ---------- Consistent metric helper functions ----------
def smape(y_true, y_pred):
    """Symmetric mean absolute percentage error"""
    denom = (np.abs(y_true) + np.abs(y_pred))
    # avoid divide by zero
    mask = denom == 0
    denom[mask] = 1e-8
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def model_file_size_mb(path):
    """Return file size in MB (if exists)"""
    try:
        return os.path.getsize(path) / (1024*1024)
    except Exception:
        return np.nan

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan

# ---------- Logging containers ----------
metrics_log = {}   # will store runtime & size & performance per model/regime
train_timers = {}
model_sizes = {}
performance_summary = {}  # global metrics

# ---------- 1) Load data ----------
print("Loading data from:", DATA_PATH)
df = read_table(DATA_PATH)
print("Raw columns:", df.columns.tolist())
df.columns = [c.strip() for c in df.columns]

# If DATETIME_COL missing, try to auto-detect
if DATETIME_COL not in df.columns:
    for c in df.columns:
        if "FECHA" in c.upper() or "HORA" in c.upper():
            DATETIME_COL = c
            print("Using datetime column:", DATETIME_COL)
            break

# Parse datetime
df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], dayfirst=True, errors='coerce')
if df[DATETIME_COL].isna().any():
    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL].astype(str), dayfirst=True, errors='coerce')
assert df[DATETIME_COL].notna().all(), "Some datetime values could not be parsed. Check your FECHA_HORA column."

df = df.sort_values(DATETIME_COL).reset_index(drop=True)

# rename simple columns if user uses longer names (common in their sample)
rename_map = {}
for col in df.columns:
    c = col.strip()
    if c.lower().startswith("dd"):
        rename_map[col] = "DD"
    if "NO2" in c.upper():
        rename_map[col] = "NO2"
    if "O3" == c.upper() or c.upper().endswith("-O3") or c.upper().startswith("O3"):
        rename_map[col] = "O3"
    if "PM10" in c.upper():
        rename_map[col] = "PM10"
    if "TMP" in c.upper() or "TEMP" in c.upper():
        rename_map[col] = "TMP"
    if "VV" in c.upper():
        rename_map[col] = "VV"
if rename_map:
    df = df.rename(columns=rename_map)
print("After renaming, columns:", df.columns.tolist())

assert TARGET_COL in df.columns, f"{TARGET_COL} not found in data columns."

# ---------- 2) Feature engineering ----------
df = create_time_features(df, DATETIME_COL)

# numeric columns used in clustering and modeling (only keep existing)
model_cols = ["DD", "NO2", "PM10", "TMP", "VV"]
model_cols = [c for c in model_cols if c in df.columns]

# create lags for O3 and main predictors
lags = [1,2,3,24]   # 1,2,3 hour and 24-hour lag
lags_for = [TARGET_COL] + model_cols
df = make_lags(df, lags_for, lags)
df = df.dropna().reset_index(drop=True)
print("Data shape after lags and dropna:", df.shape)

# ---------- 3) Discover regimes via clustering (GMM) - TRAIN-ONLY, with smoothing & validation ----------
cluster_features = [c for c in model_cols + ['hour_sin','hour_cos','month_sin','month_cos'] if c in df.columns]
print("Cluster features:", cluster_features)

# We'll split chronologically first to avoid leakage for the GMM selection
test_fraction = 0.2
n_test = int(len(df) * test_fraction)
train_df_all, test_df_all = train_test_chrono_split(df, test_size_hours=n_test)
print("Clustering will be fit on train only. Train rows:", len(train_df_all), "Test rows:", len(test_df_all))

# Fit scaler on train clustering features only
scaler_cluster = StandardScaler().fit(train_df_all[cluster_features].values)
Xc_train = scaler_cluster.transform(train_df_all[cluster_features].values)
Xc_test = scaler_cluster.transform(test_df_all[cluster_features].values)

# Choose number of clusters using BIC on train only
def select_gmm_k_on_train(X_train, k_min=2, k_max=6, random_state=RANDOM_SEED):
    best_bic = np.inf
    best_k = k_min
    best_model = None
    for k in range(k_min, k_max+1):
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=random_state, n_init=5)
        gmm.fit(X_train)
        bic = gmm.bic(X_train)
        print(f"GMM k={k} BIC={bic:.1f}")
        if bic < best_bic:
            best_bic = bic
            best_k = k
            best_model = gmm
    return best_k, best_model

print("Selecting number of regimes (GMM) on train only...")
best_k, gmm = select_gmm_k_on_train(Xc_train, k_min=2, k_max=5)
print("Selected regimes (k):", best_k)

# Assign regimes: predict on train & test using scaler and trained GMM
train_df_all['regime'] = gmm.predict(Xc_train)
test_df_all['regime'] = gmm.predict(Xc_test)

# Optional temporal smoothing to enforce regime persistence (small window)
SMOOTH_REGIMES = True
SMOOTH_WINDOW = 3  # odd integer
if SMOOTH_REGIMES:
    train_df_all['regime'] = train_df_all['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)
    test_df_all['regime'] = test_df_all['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)

# Compute clustering validity metrics on train
try:
    sil = silhouette_score(Xc_train, train_df_all['regime'])
    dbi = davies_bouldin_score(Xc_train, train_df_all['regime'])
    print(f"Clustering validity (train): Silhouette = {sil:.4f}, Davies-Bouldin = {dbi:.4f}")
    metrics_log['clustering'] = {'silhouette': float(sil), 'davies_bouldin': float(dbi)}
except Exception as e:
    print("Could not compute silhouette/DBI:", e)

# Create full df regimes by applying scaler and gmm predict to entire df (no refit)
Xc_full = scaler_cluster.transform(df[cluster_features].values)
df['regime'] = gmm.predict(Xc_full)
if SMOOTH_REGIMES:
    df['regime'] = df['regime'].rolling(SMOOTH_WINDOW, center=True, min_periods=1).median().astype(int)

# Regime summary table (means)
regime_summary = df.groupby('regime')[cluster_features].mean().round(3)
print("Regime summary (means):")
display(regime_summary)

# plot regime counts
plt.figure(figsize=(6,3))
df['regime'].value_counts().sort_index().plot(kind='bar', title='Regime counts')
plt.show()

# Persist cluster scaler & gmm
joblib.dump(scaler_cluster, os.path.join(RESULTS_DIR, "scaler_cluster.pkl"))
joblib.dump(gmm, os.path.join(RESULTS_DIR, "gmm_regimes.pkl"))

# ---------- 4) Chronological train/test split for full modelling ----------
# We'll now use train_df_all (already computed) and test_df_all for training models
train_df = train_df_all.copy().reset_index(drop=True)
test_df = test_df_all.copy().reset_index(drop=True)
print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
metrics_log['data_split'] = {'n_total': int(len(df)), 'n_train': int(len(train_df)), 'n_test': int(len(test_df)),
                            'train_fraction': float(len(train_df)/len(df)), 'test_fraction': float(len(test_df)/len(df))}

# ---------- 5) Per-regime model training ----------
# Common feature set for tabular models
tab_features = []
# features include lags for each variable in lags_for (like O3_lag1...) plus meteorological/time features
for c in df.columns:
    # if column starts with any lag prefix or matches meta features
    if any([c.startswith(f"{col}_lag") for col in lags_for]) or c in model_cols + ['hour','weekday','month','hour_sin','hour_cos','month_sin','month_cos']:
        tab_features.append(c)
# ensure target not in features
tab_features = [c for c in tab_features if c != TARGET_COL and c != DATETIME_COL and c != 'regime']
print("Tabular features used (sample):", tab_features[:30])

# Setup storage
regime_models = {}
regimes_unique = sorted(train_df['regime'].unique())
print("Regimes to train:", regimes_unique)

# ---------- 5.A) Train Random Forest per regime WITH OPTUNA ----------
rf_total_train_time = 0.0
rf_regime_times = {}

def fit_rf_for_regime_optuna(Xtrain, ytrain, Xval, yval, n_trials=20, seed=RANDOM_SEED):
    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 50, 300)
        max_depth = trial.suggest_int("max_depth", 5, 50)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=seed,
            n_jobs=-1
        )
        
        rf.fit(Xtrain, ytrain)
        preds = rf.predict(Xval)
        return mean_squared_error(yval, preds, squared=False)  # RMSE

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    # Fit final model with best params on combined train+val
    best_params = study.best_params
    final_rf = RandomForestRegressor(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        max_features=best_params['max_features'],
        random_state=seed,
        n_jobs=-1
    )
    
    # Combine train and validation for final training
    X_combined = np.vstack([Xtrain, Xval])
    y_combined = np.concatenate([ytrain, yval])
    final_rf.fit(X_combined, y_combined)
    
    return final_rf, study

for r in regimes_unique:
    print("Training Random Forest for regime:", r)
    start_r = time.time()
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < 120:
        print(f"Regime {r} has too few samples ({len(df_r)}). Skipping per-regime RF.")
        continue
        
    split_idx = int(0.8 * len(df_r))
    train_r = df_r.iloc[:split_idx]
    val_r = df_r.iloc[split_idx:]
    
    Xtrain = train_r[tab_features].values
    ytrain = train_r[TARGET_COL].values
    Xval = val_r[tab_features].values
    yval = val_r[TARGET_COL].values
    
    scaler = StandardScaler().fit(np.vstack([Xtrain, Xval]))
    Xtrain_s = scaler.transform(Xtrain)
    Xval_s = scaler.transform(Xval)
    
    print("  Tuning Random Forest (Optuna)...")
    try:
        t0 = time.time()
        rf_model, study = fit_rf_for_regime_optuna(Xtrain_s, ytrain, Xval_s, yval, n_trials=20)
        t1 = time.time()
        elapsed = t1 - t0
        rf_total_train_time += elapsed
        rf_regime_times[r] = elapsed
        
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['rf_model'] = rf_model
        regime_models[r]['rf_scaler'] = scaler
        
        # Save model and record size
        rf_path = os.path.join(RESULTS_DIR, f"rf_regime_{r}.pkl")
        joblib.dump(rf_model, rf_path)
        size_mb = model_file_size_mb(rf_path)
        model_sizes[f"rf_regime_{r}"] = size_mb
        train_timers[f"rf_regime_{r}"] = elapsed
        
        print(f"  Random Forest done. Best RMSE (val): {study.best_value:.4f}, train time(s): {elapsed:.2f}, size(MB): {size_mb:.3f}")
    except Exception as e:
        print("  Random Forest training failed for regime", r, "due to:", e)

# Train global fallback Random Forest with Optuna
print("Training global fallback Random Forest on whole train set...")
rf_global = None
scaler_rf_glob = None
rf_global_train_time = None

try:
    t0_glob = time.time()
    split_idx = int(0.9 * len(train_df))
    train_glob = train_df.iloc[:split_idx]
    val_glob = train_df.iloc[split_idx:]
    
    Xtrain_g = train_glob[tab_features].values
    ytrain_g = train_glob[TARGET_COL].values
    Xval_g = val_glob[tab_features].values
    yval_g = val_glob[TARGET_COL].values
    
    scaler_rf_glob = StandardScaler().fit(np.vstack([Xtrain_g, Xval_g]))
    Xtrain_gs = scaler_rf_glob.transform(Xtrain_g)
    Xval_gs = scaler_rf_glob.transform(Xval_g)
    
    rf_global, study_glob = fit_rf_for_regime_optuna(Xtrain_gs, ytrain_g, Xval_gs, yval_g, n_trials=25)
    t1_glob = time.time()
    rf_global_train_time = t1_glob - t0_glob
    
    rf_global_path = os.path.join(RESULTS_DIR, "rf_global.pkl")
    joblib.dump(rf_global, rf_global_path)
    joblib.dump(scaler_rf_glob, os.path.join(RESULTS_DIR, "scaler_rf_global.pkl"))
    joblib.dump(study_glob, os.path.join(RESULTS_DIR, "optuna_rf_global.pkl"))
    
    size_glob = model_file_size_mb(rf_global_path)
    model_sizes["rf_global"] = size_glob
    train_timers["rf_global"] = rf_global_train_time
    
    print(f"Global RF trained. Best RMSE: {study_glob.best_value:.4f}, time(s): {rf_global_train_time:.2f}, size(MB): {size_glob:.3f}")
except Exception as e:
    print("Global Random Forest training failed:", e)

# ---------- 5.B) Train LightGBM per regime with Optuna ----------
def fit_lgb_for_regime(Xtrain, ytrain, Xval, yval, n_trials=25, seed=RANDOM_SEED):
    def objective(trial):
        param = {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "seed": seed,
            "num_leaves": trial.suggest_int("num_leaves", 16, 128),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        }

        dtrain = lgb.Dataset(Xtrain, label=ytrain)
        dvalid = lgb.Dataset(Xval, label=yval)

        bst = lgb.train(
            param,
            dtrain,
            valid_sets=[dvalid],
            num_boost_round=1000,
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(0)
            ]
        )

        preds = bst.predict(Xval, num_iteration=bst.best_iteration)
        return mean_squared_error(yval, preds, squared=False)  # RMSE

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    # Fit final model on combined train+val using best params
    best_params = study.best_params
    best_params.update({
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "seed": seed
    })

    dtrain_all = lgb.Dataset(np.vstack([Xtrain, Xval]), label=np.concatenate([ytrain, yval]))
    final = lgb.train(
        best_params,
        dtrain_all,
        num_boost_round=200  # keep reasonable
    )

    # store iteration info
    study.best_trial.user_attrs["best_iteration"] = final.current_iteration() if hasattr(final, "current_iteration") else 200
    return final, study

# Timing variables for LGB training
lgb_total_train_time = 0.0
lgb_regime_times = {}

for r in regimes_unique:
    print("Training LGB for regime:", r)
    start_r = time.time()
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < 120:
        print(f"Regime {r} has too few samples ({len(df_r)}). Skipping per-regime LGB (will use global fallback later).")
        continue
    split_idx = int(0.8 * len(df_r))
    train_r = df_r.iloc[:split_idx]
    val_r = df_r.iloc[split_idx:]
    Xtrain = train_r[tab_features].values
    ytrain = train_r[TARGET_COL].values
    Xval = val_r[tab_features].values
    yval = val_r[TARGET_COL].values
    scaler = StandardScaler().fit(np.vstack([Xtrain, Xval]))
    Xtrain_s = scaler.transform(Xtrain)
    Xval_s = scaler.transform(Xval)
    print("  Tuning LightGBM (Optuna)...")
    try:
        t0 = time.time()
        lgb_model, study = fit_lgb_for_regime(Xtrain_s, ytrain, Xval_s, yval, n_trials=25)
        t1 = time.time()
        elapsed = t1 - t0
        lgb_total_train_time += elapsed
        lgb_regime_times[r] = elapsed
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['lgb_model'] = lgb_model
        regime_models[r]['lgb_scaler'] = scaler
        joblib.dump(study, os.path.join(RESULTS_DIR, f"optuna_lgb_regime_{r}.pkl"))
        # Save model and record size
        lgb_path = os.path.join(RESULTS_DIR, f"lgb_regime_{r}.txt")
        lgb_model.save_model(lgb_path)
        size_mb = model_file_size_mb(lgb_path)
        model_sizes[f"lgb_regime_{r}"] = size_mb
        train_timers[f"lgb_regime_{r}"] = elapsed
        print("  LightGBM done. best RMSE (val):", study.best_value, "train time(s):", round(elapsed,2), "size(MB):", round(size_mb,3))
    except Exception as e:
        print("  LightGBM training failed for regime", r, "due to:", e)

# Train a global fallback LightGBM on all train data for regimes with too few samples
print("Training global fallback LightGBM on whole train set (for small regimes)...")
lgb_global = None
scaler_glob = None
lgb_global_train_time = None
try:
    t0_glob = time.time()
    split_idx = int(0.9 * len(train_df))
    train_glob = train_df.iloc[:split_idx]
    val_glob = train_df.iloc[split_idx:]
    Xtrain_g = train_glob[tab_features].values
    ytrain_g = train_glob[TARGET_COL].values
    Xval_g = val_glob[tab_features].values
    yval_g = val_glob[TARGET_COL].values
    scaler_glob = StandardScaler().fit(np.vstack([Xtrain_g, Xval_g]))
    Xtrain_gs = scaler_glob.transform(Xtrain_g)
    Xval_gs = scaler_glob.transform(Xval_g)
    lgb_global, study_glob = fit_lgb_for_regime(Xtrain_gs, ytrain_g, Xval_gs, yval_g, n_trials=20)
    t1_glob = time.time()
    lgb_global_train_time = t1_glob - t0_glob
    joblib.dump(lgb_global, os.path.join(RESULTS_DIR, "lgb_global.pkl"))
    joblib.dump(scaler_glob, os.path.join(RESULTS_DIR, "scaler_lgb_global.pkl"))
    # save text model
    lgb_global_path = os.path.join(RESULTS_DIR, "lgb_global.txt")
    lgb_global.save_model(lgb_global_path)
    size_glob = model_file_size_mb(lgb_global_path)
    model_sizes["lgb_global"] = size_glob
    train_timers["lgb_global"] = lgb_global_train_time
    print("Global LGB trained. best RMSE:", study_glob.best_value, "time(s):", round(lgb_global_train_time,2), "size(MB):", round(size_glob,3))
except Exception as e:
    print("Global LightGBM training failed:", e)

# ---------- 5.C) Train small LSTM per regime ----------
SEQ_LEN = 24  # use last 24 hours to predict next hour

def make_sequences(df_reg, seq_len, feature_cols, target_col):
    Xs, ys = [], []
    arr = df_reg[feature_cols].values
    tar = df_reg[target_col].values
    for i in range(seq_len, len(df_reg)):
        Xs.append(arr[i-seq_len:i])
        ys.append(tar[i])
    return np.array(Xs), np.array(ys)

lstm_histories = {}
lstm_train_times = {}
lstm_total_time = 0.0

for r in regimes_unique:
    print("Training LSTM for regime:", r)
    df_r = train_df[train_df['regime']==r].copy().reset_index(drop=True)
    if len(df_r) < SEQ_LEN + 30:
        print(f"  Too few samples ({len(df_r)}). Skipping LSTM for regime {r}.")
        continue
    seq_features = [c for c in tab_features]  # use same tab features in sequence
    scaler_seq = MinMaxScaler()
    scaler_seq.fit(df_r[seq_features].values)
    arr_scaled = scaler_seq.transform(df_r[seq_features].values)
    df_r_scaled = df_r.copy()
    df_r_scaled[seq_features] = arr_scaled
    Xs, ys = make_sequences(df_r_scaled, SEQ_LEN, seq_features, TARGET_COL)
    split = int(0.8 * len(Xs))
    Xtrain, Xval = Xs[:split], Xs[split:]
    ytrain, yval = ys[:split], ys[split:]
    def build_lstm_model(n_features, lr=1e-3, units=64, dropout=0.2):
        inp = keras.Input(shape=(SEQ_LEN, n_features))
        x = layers.LSTM(units, return_sequences=False)(inp)
        x = layers.Dropout(dropout)(x)
        out = layers.Dense(1)(x)
        model = keras.Model(inputs=inp, outputs=out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss='mse')
        return model
    def optuna_lstm_objective(trial):
        units = trial.suggest_int("units", 16, 128)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
        model = build_lstm_model(Xtrain.shape[2], lr=lr, units=units, dropout=dropout)
        early = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')
        h = model.fit(Xtrain, ytrain, validation_data=(Xval, yval), epochs=60, batch_size=32, callbacks=[early], verbose=0)
        val_rmse = np.sqrt(min(h.history['val_loss']))
        return val_rmse
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    n_trials = 20
    try:
        t0 = time.time()
        study.optimize(optuna_lstm_objective, n_trials=n_trials, show_progress_bar=False)
        t1 = time.time()
        elapsed = t1 - t0
        lstm_total_time += elapsed
        lstm_train_times[r] = elapsed
        print("  Best LSTM RMSE (val):", study.best_value, "t(s):", round(elapsed,2))
        best = study.best_params
        final_model = build_lstm_model(Xs.shape[2], lr=best.get('lr',1e-3), units=best.get('units',64), dropout=best.get('dropout',0.2))
        early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='loss')
        t0f = time.time()
        final_model.fit(Xs, ys, epochs=100, batch_size=32, callbacks=[early], verbose=0)
        t1f = time.time()
        fit_elapsed = t1f - t0f
        total_elapsed = elapsed + fit_elapsed
        lstm_train_times[r] = total_elapsed
        lstm_histories[r] = study.best_value
        regime_models[r] = regime_models.get(r, {})
        regime_models[r]['lstm_model'] = final_model
        regime_models[r]['lstm_scaler'] = scaler_seq
        # save model (h5) and size
        lstm_path = os.path.join(RESULTS_DIR, f"lstm_regime_{r}.h5")
        try:
            final_model.save(lstm_path)
        except Exception:
            # fallback to pickle for keras (may not be ideal)
            with open(os.path.join(RESULTS_DIR, f"lstm_regime_{r}.pkl"), "wb") as f:
                pickle.dump(final_model, f)
            lstm_path = os.path.join(RESULTS_DIR, f"lstm_regime_{r}.pkl")
        size_mb = model_file_size_mb(lstm_path)
        model_sizes[f"lstm_regime_{r}"] = size_mb
        train_timers[f"lstm_regime_{r}"] = total_elapsed
        joblib.dump(study, os.path.join(RESULTS_DIR, f"optuna_lstm_regime_{r}.pkl"))
        print(f"  Final LSTM fit time(s): {round(fit_elapsed,2)}, total(s): {round(total_elapsed,2)}, size(MB): {round(size_mb,3)}")
    except Exception as e:
        print("  LSTM training failed for regime", r, "due to:", e)

# ---------- 6) Train regime classifier (meta-controller) ----------
# The classifier predicts the regime at time t given current features (trained on train set only)
clf_features = cluster_features + [TARGET_COL] + [f"{TARGET_COL}_lag{l}" for l in lags]
clf_features = [c for c in clf_features if c in train_df.columns]
print("Regime classifier features:", clf_features)

Xclf = train_df[clf_features].values
yclf = train_df['regime'].values
clf_scaler = StandardScaler().fit(Xclf)
Xclf_s = clf_scaler.transform(Xclf)
t0_clf = time.time()
rfc = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED)
rfc.fit(Xclf_s, yclf)
t1_clf = time.time()
clf_train_time = t1_clf - t0_clf
joblib.dump(rfc, os.path.join(RESULTS_DIR, "regime_classifier.pkl"))
joblib.dump(clf_scaler, os.path.join(RESULTS_DIR, "regime_clf_scaler.pkl"))
# record size/time
rfc_path = os.path.join(RESULTS_DIR, "regime_classifier.pkl")
model_sizes["regime_classifier_rf"] = model_file_size_mb(rfc_path)
train_timers["regime_classifier_rf"] = clf_train_time
print("Regime classifier trained. Train accuracy (on scaled train):", rfc.score(Xclf_s, yclf), "time(s):", round(clf_train_time,2), "size(MB):", round(model_sizes["regime_classifier_rf"],3))

# ---------- 7) Attention-style meta-blender ----------
from tensorflow.keras import backend as K

def build_blender(input_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    w = layers.Dense(2, activation='softmax')(x)  # outputs two weights summing to 1
    model = keras.Model(inputs=inp, outputs=w)
    # we'll use custom training loop below
    return model

# Prepare blender training data: use last 10% of train_df (chronological)
blend_df = train_df.iloc[int(0.9*len(train_df)):].copy().reset_index(drop=True)
blend_features = tab_features  # use tab features as input to blender
X_blend = []
Y_blend = []
preds_lgb_all = []
preds_lstm_all = []

# We'll use the global fallback LGB/scaler if per-regime LGB missing
lgb_global_exists = os.path.exists(os.path.join(RESULTS_DIR, "lgb_global.pkl"))
if lgb_global_exists:
    try:
        lgb_global = joblib.load(os.path.join(RESULTS_DIR, "lgb_global.pkl"))
        scaler_glob = joblib.load(os.path.join(RESULTS_DIR, "scaler_lgb_global.pkl"))
    except:
        lgb_global = None
        scaler_glob = None
else:
    lgb_global = None
    scaler_glob = None

# Check if RF global exists
rf_global_exists = os.path.exists(os.path.join(RESULTS_DIR, "rf_global.pkl"))
if rf_global_exists:
    try:
        rf_global = joblib.load(os.path.join(RESULTS_DIR, "rf_global.pkl"))
        scaler_rf_glob = joblib.load(os.path.join(RESULTS_DIR, "scaler_rf_global.pkl"))
    except:
        rf_global = None
        scaler_rf_glob = None
else:
    rf_global = None
    scaler_rf_glob = None

for idx, row in blend_df.reset_index(drop=True).iterrows():
    r = int(row['regime'])
    # LGB prediction
    if 'lgb_model' in regime_models.get(r, {}):
        scaler_lgb = regime_models[r]['lgb_scaler']
        lgb_model = regime_models[r]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_global.predict(X_row)[0])
    else:
        pred_lgb = float(row[TARGET_COL])  # fallback to persistence

    # LSTM prediction: build sequence from training data only (NO DATA LEAKAGE)
    if 'lstm_model' in regime_models.get(r, {}):
        lstm_model = regime_models[r]['lstm_model']
        scaler_seq = regime_models[r]['lstm_scaler']
        dt = row[DATETIME_COL]
        # Use training data only for LSTM sequences - NO DATA LEAKAGE
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==r)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values)
            seq = seq.reshape(1, SEQ_LEN, len(tab_features))
            try:
                pred_lstm = float(lstm_model.predict(seq)[0,0])
            except:
                pred_lstm = pred_lgb
        else:
            pred_lstm = pred_lgb
    else:
        pred_lstm = pred_lgb

    preds_lgb_all.append(pred_lgb)
    preds_lstm_all.append(pred_lstm)
    X_blend.append(row[blend_features].values)
    Y_blend.append(row[TARGET_COL])

X_blend = np.array(X_blend)
Y_blend = np.array(Y_blend)
preds_lgb_all = np.array(preds_lgb_all)
preds_lstm_all = np.array(preds_lstm_all)

scaler_blend = StandardScaler().fit(X_blend)
X_blend_scaled = scaler_blend.transform(X_blend)
X_blend_in = np.hstack([X_blend_scaled, preds_lgb_all.reshape(-1,1), preds_lstm_all.reshape(-1,1)])
blender_input_dim = X_blend_in.shape[1]
print("Blender input dim:", blender_input_dim)

def blender_train_end_to_end(Xin, predsA, predsB, Y, batch=32, epochs=60):
    # Build model that outputs weights; we'll train with a custom loop minimizing MSE between weighted preds and Y
    inp = keras.Input(shape=(Xin.shape[1],))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    w = layers.Dense(2, activation='softmax')(x)
    model = keras.Model(inputs=inp, outputs=w)
    optimizer = keras.optimizers.Adam()
    # prepare dataset
    N = len(Xin)
    for epoch in range(epochs):
        idxs = np.arange(N)
        np.random.shuffle(idxs)
        epoch_losses = []
        for i in range(0, N, batch):
            batch_idx = idxs[i:i+batch]
            Xin_b = Xin[batch_idx]
            predsA_b = predsA[batch_idx]
            predsB_b = predsB[batch_idx]
            Y_b = Y[batch_idx]
            with tf.GradientTape() as tape:
                W_b = model(Xin_b, training=True)  # shape (b,2)
                combined = W_b[:,0]*predsA_b + W_b[:,1]*predsB_b
                loss = tf.reduce_mean(tf.square(combined - Y_b))
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            epoch_losses.append(float(loss))
        if epoch % 10 == 0 or epoch == epochs-1:
            print(f"Blender epoch {epoch}, loss {np.mean(epoch_losses):.6f}")
    return model

print("Training blender network (end-to-end)...")
t0_blend = time.time()
blender_model = blender_train_end_to_end(X_blend_in, preds_lgb_all, preds_lstm_all, Y_blend, batch=32, epochs=60)
t1_blend = time.time()
blender_train_time = t1_blend - t0_blend
# save blender & scaler
joblib.dump(scaler_blend, os.path.join(RESULTS_DIR, "scaler_blend.pkl"))
blender_path = os.path.join(RESULTS_DIR, "blender_model.h5")
try:
    blender_model.save(blender_path)
except Exception:
    # fallback to pickle
    with open(os.path.join(RESULTS_DIR, "blender_model.pkl"), "wb") as f:
        pickle.dump(blender_model, f)
    blender_path = os.path.join(RESULTS_DIR, "blender_model.pkl")
model_sizes["blender_model"] = model_file_size_mb(blender_path)
train_timers["blender_model"] = blender_train_time
print("Blender trained. time(s):", round(blender_train_time,2), "size(MB):", round(model_sizes["blender_model"],3))

# ---------- 8) Prediction pipeline on test set ----------
def predict_one_row(row):
    """Given a row from test_df, predict O3 for next hour using DMOFF pipeline."""
    # predict regime for current row using regime classifier (trained on train only)
    xf = row[clf_features].values.reshape(1,-1)
    xf_s = clf_scaler.transform(xf)
    pred_reg = int(rfc.predict(xf_s)[0])

    # LGB prediction (per-regime or global fallback)
    if 'lgb_model' in regime_models.get(pred_reg, {}):
        scaler_lgb = regime_models[pred_reg]['lgb_scaler']
        lgb_model = regime_models[pred_reg]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        pred_lgb = float(lgb_global.predict(X_row)[0])
    else:
        pred_lgb = float(row[TARGET_COL])

    # LSTM prediction - USE TRAINING DATA ONLY (NO DATA LEAKAGE)
    if 'lstm_model' in regime_models.get(pred_reg, {}):
        lstm_model = regime_models[pred_reg]['lstm_model']
        scaler_seq = regime_models[pred_reg]['lstm_scaler']
        dt = row[DATETIME_COL]
        # Use training data only for LSTM sequences - NO DATA LEAKAGE
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==pred_reg)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values).reshape(1, SEQ_LEN, len(tab_features))
            try:
                pred_lstm = float(lstm_model.predict(seq)[0,0])
            except:
                pred_lstm = pred_lgb
        else:
            pred_lstm = pred_lgb
    else:
        pred_lstm = pred_lgb

    # Build blender input
    Xb_raw = row[blend_features].values.reshape(1,-1)
    Xb_scaled = scaler_blend.transform(Xb_raw)
    Xb_in = np.hstack([Xb_scaled, np.array([[pred_lgb, pred_lstm]])])
    w = blender_model.predict(Xb_in)[0]  # two weights
    pred = w[0]*pred_lgb + w[1]*pred_lstm
    return pred, pred_reg, pred_lgb, pred_lstm, w

# Run predictions across test set with timing
y_true = []
y_pred = []
pred_details = []
pred_times = []
start_all_preds = time.time()
for idx, row in test_df.iterrows():
    t0p = time.time()
    p, preg, plgb, plstm, w = predict_one_row(row)
    t1p = time.time()
    pred_times.append(t1p - t0p)
    y_true.append(row[TARGET_COL])
    y_pred.append(p)
    pred_details.append((preg, plgb, plstm, w))
end_all_preds = time.time()
total_pred_time = end_all_preds - start_all_preds
avg_pred_time = np.mean(pred_times) if pred_times else np.nan
pred_percentiles = np.percentile(pred_times, [50, 90, 95]) if pred_times else [np.nan, np.nan, np.nan]
metrics_log['prediction_time'] = {'total_s': float(total_pred_time), 'avg_per_sample_s': float(avg_pred_time),
                                 'p50_s': float(pred_percentiles[0]), 'p90_s': float(pred_percentiles[1]), 'p95_s': float(pred_percentiles[2])}

y_true = np.array(y_true); y_pred = np.array(y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
smape_val = smape(y_true, y_pred)
r2 = safe_r2(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}, MAE: {mae:.3f}, sMAPE%: {smape_val:.3f}, R2: {r2:.3f}")

# Save predictions
test_df_out = test_df.copy()
test_df_out['pred_dmoff'] = y_pred
test_df_out['obs'] = y_true
test_df_out.to_csv(os.path.join(RESULTS_DIR, "test_predictions_dmoff.csv"), index=False)

# ---------- 9) COMPREHENSIVE MODEL COMPARISON WITH CONSISTENT METRICS ----------
print("\n" + "="*80)
print("COMPREHENSIVE MODEL PERFORMANCE COMPARISON")
print("="*80)

# Store individual model predictions
test_df_out['pred_rf_only'] = np.nan
test_df_out['pred_lgb_only'] = np.nan
test_df_out['pred_lstm_only'] = np.nan
test_df_out['pred_regime'] = np.nan

# Get individual model predictions for ALL models
for idx, row in test_df_out.iterrows():
    r = int(row['regime'])
    
    # Random Forest prediction
    if 'rf_model' in regime_models.get(r, {}):
        scaler_rf = regime_models[r]['rf_scaler']
        rf_model = regime_models[r]['rf_model']
        X_row = scaler_rf.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_rf_only'] = float(rf_model.predict(X_row)[0])
    elif rf_global is not None:
        X_row = scaler_rf_glob.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_rf_only'] = float(rf_global.predict(X_row)[0])
    
    # LightGBM only prediction
    if 'lgb_model' in regime_models.get(r, {}):
        scaler_lgb = regime_models[r]['lgb_scaler']
        lgb_model = regime_models[r]['lgb_model']
        X_row = scaler_lgb.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_lgb_only'] = float(lgb_model.predict(X_row)[0])
    elif lgb_global is not None:
        X_row = scaler_glob.transform(row[tab_features].values.reshape(1,-1))
        test_df_out.at[idx, 'pred_lgb_only'] = float(lgb_global.predict(X_row)[0])
    
    # LSTM only prediction (with the data leakage fix!)
    if 'lstm_model' in regime_models.get(r, {}):
        lstm_model = regime_models[r]['lstm_model']
        scaler_seq = regime_models[r]['lstm_scaler']
        dt = row[DATETIME_COL]
        window = train_df[(train_df[DATETIME_COL] < dt) & (train_df['regime']==r)].tail(SEQ_LEN)
        if len(window) == SEQ_LEN:
            seq = scaler_seq.transform(window[tab_features].values).reshape(1, SEQ_LEN, len(tab_features))
            try:
                test_df_out.at[idx, 'pred_lstm_only'] = float(lstm_model.predict(seq)[0,0])
            except:
                test_df_out.at[idx, 'pred_lstm_only'] = test_df_out.at[idx, 'pred_lgb_only']
    
    # Store the predicted regime
    test_df_out.at[idx, 'pred_regime'] = r

# Calculate comprehensive metrics for each model - GLOBAL LEVEL
models_comparison = []

# Global performance for each model
for model_name, pred_col in [('DMOFF (Hybrid)', 'pred_dmoff'), 
                            ('Random Forest', 'pred_rf_only'), 
                            ('LightGBM Only', 'pred_lgb_only'), 
                            ('LSTM Only', 'pred_lstm_only')]:
    
    if pred_col not in test_df_out.columns or test_df_out[pred_col].isna().all():
        continue
        
    mask = test_df_out[pred_col].notna()
    y_true_model = test_df_out.loc[mask, 'obs'].values
    y_pred_model = test_df_out.loc[mask, pred_col].values
    
    if len(y_true_model) > 0:
        rmse_m = np.sqrt(mean_squared_error(y_true_model, y_pred_model))
        mae_m = mean_absolute_error(y_true_model, y_pred_model)
        r2_m = safe_r2(y_true_model, y_pred_model)
        smape_m = smape(y_true_model, y_pred_model)
        
        models_comparison.append({
            'model': model_name,
            'n_samples': len(y_true_model),
            'rmse': rmse_m,
            'mae': mae_m,
            'r2': r2_m,
            'smape_pct': smape_m
        })
        
        print(f"{model_name:20} | N: {len(y_true_model):5} | RMSE: {rmse_m:.3f} | MAE: {mae_m:.3f} | R²: {r2_m:.3f} | sMAPE: {smape_m:.3f}%")

# Create a comprehensive comparison table
print("\n" + "="*100)
print("GLOBAL PERFORMANCE COMPARISON TABLE")
print("="*100)
comparison_df = pd.DataFrame(models_comparison)
if not comparison_df.empty:
    display(comparison_df.round(4))

# ---------- 10) CONSISTENT PER-REGIME PERFORMANCE COMPARISON ----------
print("\n" + "="*80)
print("CONSISTENT PER-REGIME PERFORMANCE COMPARISON")
print("="*80)

per_regime_comprehensive = []
for r in sorted(test_df_out['regime'].unique()):
    df_r = test_df_out[test_df_out['regime']==r]
    if len(df_r) == 0:
        continue
        
    regime_data = {'regime': int(r), 'n_samples': len(df_r)}
    
    # Calculate consistent metrics for each model in this regime
    for model_name, pred_col in [('DMOFF', 'pred_dmoff'), 
                                ('Random Forest', 'pred_rf_only'), 
                                ('LightGBM', 'pred_lgb_only'), 
                                ('LSTM', 'pred_lstm_only')]:
        
        if pred_col not in df_r.columns or df_r[pred_col].isna().all():
            continue
            
        mask = df_r[pred_col].notna()
        if mask.sum() == 0:
            continue
            
        yt = df_r.loc[mask, 'obs'].values
        yp = df_r.loc[mask, pred_col].values
        
        # Consistent metrics: RMSE, MAE, R², sMAPE
        rmse_r = np.sqrt(mean_squared_error(yt, yp))
        mae_r = mean_absolute_error(yt, yp)
        r2_r = safe_r2(yt, yp)
        smape_r = smape(yt, yp)
        
        # Store with model prefix
        regime_data[f'{model_name}_rmse'] = float(rmse_r)
        regime_data[f'{model_name}_mae'] = float(mae_r)
        regime_data[f'{model_name}_r2'] = float(r2_r) if not np.isnan(r2_r) else None
        regime_data[f'{model_name}_smape_pct'] = float(smape_r)
    
    per_regime_comprehensive.append(regime_data)

# Create comprehensive per-regime dataframe
per_regime_df = pd.DataFrame(per_regime_comprehensive)

# Display per-regime results with consistent metrics
print("\nDetailed Per-Regime Performance (Consistent Metrics):")
print("-" * 120)
for _, row in per_regime_df.iterrows():
    print(f"\nRegime {row['regime']} (N={row['n_samples']}):")
    print("-" * 80)
    
    for model_name in ['DMOFF', 'Random Forest', 'LightGBM', 'LSTM']:
        rmse_key = f'{model_name}_rmse'
        if rmse_key in row and not pd.isna(row[rmse_key]):
            print(f"  {model_name:15} | "
                  f"RMSE: {row[rmse_key]:.3f} | "
                  f"MAE: {row[f'{model_name}_mae']:.3f} | "
                  f"R²: {row[f'{model_name}_r2']:.3f} | "
                  f"sMAPE: {row[f'{model_name}_smape_pct']:.3f}%")

# Find best model for each regime
print("\n" + "="*80)
print("BEST MODEL IN EACH REGIME")
print("="*80)
for _, row in per_regime_df.iterrows():
    regime = row['regime']
    best_rmse = float('inf')
    best_model = None
    
    for model_name in ['DMOFF', 'Random Forest', 'LightGBM', 'LSTM']:
        rmse_key = f'{model_name}_rmse'
        if rmse_key in row and not pd.isna(row[rmse_key]):
            if row[rmse_key] < best_rmse:
                best_rmse = row[rmse_key]
                best_model = model_name
    
    if best_model:
        print(f"Regime {regime}: {best_model} (RMSE: {best_rmse:.3f})")

# Save comprehensive comparison results
comparison_df.to_csv(os.path.join(RESULTS_DIR, "global_model_comparison_metrics.csv"), index=False)
per_regime_df.to_csv(os.path.join(RESULTS_DIR, "per_regime_comprehensive_metrics.csv"), index=False)
test_df_out.to_csv(os.path.join(RESULTS_DIR, "test_predictions_all_models.csv"), index=False)

print(f"\nGlobal comparison results saved to: global_model_comparison_metrics.csv")
print(f"Per-regime comparison results saved to: per_regime_comprehensive_metrics.csv")

# ---------- 11) Efficiency & performance summary ----------
# Collect global performance metrics with consistent metrics
performance_summary = {
    'rmse': float(rmse),
    'mae': float(mae),
    'smape_pct': float(smape_val),
    'r2': float(r2) if not np.isnan(r2) else None,
    'n_test_samples': int(len(test_df)),
    'total_prediction_time_s': float(total_pred_time),
    'avg_prediction_time_per_sample_s': float(avg_pred_time)
}

# Merge with metrics_log
metrics_log['performance'] = performance_summary
metrics_log['model_sizes_mb'] = model_sizes
metrics_log['train_times_s'] = train_timers
metrics_log['rf_per_regime_train_times'] = {str(k): float(v) for k,v in rf_regime_times.items()}
metrics_log['lgb_per_regime_train_times'] = {str(k): float(v) for k,v in lgb_regime_times.items()}
metrics_log['lstm_per_regime_train_times'] = {str(k): float(v) for k,v in lstm_train_times.items()}
metrics_log['blender_train_time_s'] = float(blender_train_time)

# Save metrics_log as JSON
with open(os.path.join(RESULTS_DIR, "metrics_log.json"), "w") as f:
    json.dump(metrics_log, f, indent=2)

# Write a consolidated CSV with global and summary info
global_metrics_df = pd.DataFrame([performance_summary])
global_metrics_df.to_csv(os.path.join(RESULTS_DIR, "global_performance_metrics.csv"), index=False)

# Save comprehensive comparison
report = {
    'global_metrics': performance_summary,
    'model_comparison': models_comparison,
    'per_regime_metrics': per_regime_comprehensive,
    'model_sizes_mb': model_sizes,
    'train_times_s': train_timers,
    'clustering_metrics': metrics_log.get('clustering', {})
}
with open(os.path.join(RESULTS_DIR, "comprehensive_dmoff_report.json"), "w") as f:
    json.dump(report, f, indent=2)

# ---------- 12) Save everything ----------
joblib.dump(regime_models, os.path.join(RESULTS_DIR, "regime_models.pkl"))
joblib.dump(rfc, os.path.join(RESULTS_DIR, "regime_classifier_rf.pkl"))
joblib.dump(scaler_cluster, os.path.join(RESULTS_DIR, "scaler_cluster_final.pkl"))
joblib.dump(gmm, os.path.join(RESULTS_DIR, "gmm_regimes_final.pkl"))
print("All models and artifacts saved in", RESULTS_DIR)

# Final comprehensive summary
print("\n" + "="*80)
print("COMPREHENSIVE SUMMARY - ALL MODELS")
print("="*80)
print(f"Dataset: {len(train_df)} train, {len(test_df)} test samples")
print(f"Regimes identified: {best_k}")
print("\nOverall Performance Ranking:")
print("-" * 60)

# Sort models by RMSE (ascending)
sorted_models = sorted(models_comparison, key=lambda x: x['rmse'])
for i, model in enumerate(sorted_models, 1):
    print(f"{i}. {model['model']:20} | RMSE: {model['rmse']:.3f} | MAE: {model['mae']:.3f} | R²: {model['r2']:.3f}")

print(f"\nDMOFF Hybrid Improvement over Best Base Model:")
if len(sorted_models) > 1:
    best_base = sorted_models[1]  # First after DMOFF (assuming DMOFF is best)
    dmoff = sorted_models[0]
    rmse_improvement = ((best_base['rmse'] - dmoff['rmse']) / best_base['rmse']) * 100
    mae_improvement = ((best_base['mae'] - dmoff['mae']) / best_base['mae']) * 100
    print(f"RMSE Improvement: {rmse_improvement:.1f}% over {best_base['model']}")
    print(f"MAE Improvement: {mae_improvement:.1f}% over {best_base['model']}")

print(f"\nTotal prediction time: {total_pred_time:.2f}s")
print(f"Average prediction time per sample: {avg_pred_time:.6f}s")
print("="*80)

# ---------- End ----------
print("\nDMOFF pipeline completed with consistent metrics comparison.")
print("All results saved to:", RESULTS_DIR)
print("Check 'global_model_comparison_metrics.csv' and 'per_regime_comprehensive_metrics.csv' for detailed results.")